# Agrupamento de Dados Sísmicos e Generalização entre Bacias Sedimentares

## Análise Comparativa dos Algoritmos K-means e Gaussian Mixture Model (GMM)

### 1. Introdução

A observação de dados sísmicos é uma das ferramentas mais importante para a caracterização de estruturas geológicas em subsurperfícies. Porém, diante dos dados trabalhados, existe o problema dos elevandos volumes de informação e a complexidade dos sinais, tornando o processo de interpretação algo bastante desafiador, motivando o uso de técnicas de Machine Learning para ajudar no processo de identificação de padrões.

Neste trabalho, são investigados dois algoritmos de aprendizado não supervisionado: K-means e Gaussian Mixture Model (GMM). A aplicação desses modelos será realizada a partir da extração de atributos sísmicos, permitindo agrupar amostras com características semelhantes e analisar sua distribuição nos dados.

Primeiramente, serão usados dados sísmicos pré-stack da Bacia de Jequitinhonha para o processo de desenvolvimento, treinamento e validação interna dos modelos. Em seguida, os modelos serão aplicados a dados de outra bacia sedimentar, a Bacia de Campos, com a finalidade de avaliar sua capacidade de generalização em diferentes contextos geológicos.

Dentre os aspectos analisados, serão aplicados a qualidade dos agrupamentos, otimização de hiperparâmetros incerteza das classificações probabilísticas e ao desempenho computacional dos algoritmos.

### 2. Importação das Bibliotecas e Configurações

Nesta etapa, são importadas as bibliotecas necessárias para
o processamento dos dados sísmicos, a extração de atributos,
a implementação dos modelos de Machine Learning e a
visualização dos resultados.

In [ ]:
# Manipulação de arquivos e sistema
import os
import sys
from pathlib import Path
import time

# Processamento numérico
import numpy as np
import pandas as pd

# Processamento de sinais sísmicos
import segyio
from scipy.signal import hilbert
from scipy.ndimage import uniform_filter1d

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Métricas de avaliação
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

# Visualização
import matplotlib.pyplot as plt

# Armazenamento de modelos
import joblib

# Monitoramento computacional
import psutil

print("Bibliotecas importadas com sucesso!")

#### 2.1 Configuração dos parâmetros gerais

In [ ]:
# Reprodutibilidade
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Separação dos dados
TRAIN_SIZE = 0.64
TUNING_SIZE = 0.16
VALIDATION_SIZE = 0.20

# Limites de amostragem
MAX_TRAIN_SAMPLES = 100_000
MAX_VALIDATION_SAMPLES = 20_000
SILHOUETTE_SAMPLES = 5_000

# Intervalo de clusters avaliados
K_RANGE = range(2, 10)

# Processamento em blocos
BATCH_SIZE = 100_000
TRACE_BATCH_SIZE = 128

# Limites para visualização
MAX_DISPLAY_TRACES = 800
MAX_DISPLAY_SAMPLES = 1500

# Configuração dos gráficos
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 100,
    "font.size": 11
})


### 3. Carregamento dos Dados Sísmicos

In [ ]:
# Pasta principal do projeto
PROJECT_ROOT = Path.cwd().parent

# Diretórios do projeto
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"

# Permitir a importação dos módulos da pasta src
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# Dados de treinamento
JEQUITINHONHA_PATH = DATA_RAW / "jequitinhonha.sgy"

# Dados de teste externo
CAMPOS_PATH = DATA_RAW / "campos.sgy"

# Verificação dos arquivos
print("Jequitinhonha:", JEQUITINHONHA_PATH.is_file())
print("Campos:", CAMPOS_PATH.is_file())

#### 3.1. Carregamento dos dados da Bacia do Jequitinhonha

In [ ]:
from src.data_loader import load_segy_manual_2d_final

seismic_data, inlines, crosslines, times = (
    load_segy_manual_2d_final(JEQUITINHONHA_PATH)
)

print("Dados do Jequitinhonha carregados!")
print("Dimensões:", seismic_data.shape)
print("Número de traços:", len(inlines))
print("Amostras por traço:", len(times))

#### 3.2. Carregamento dos dados da Bacia de Campos

In [ ]:
(
    seismic_data_campos,
    inlines_campos,
    crosslines_campos,
    times_campos
) = load_segy_manual_2d_final(CAMPOS_PATH)

print("Dados de Campos carregados!")
print("Dimensões:", seismic_data_campos.shape)
print("Número de traços:", len(inlines_campos))
print("Amostras por traço:", len(times_campos))

#### 3.3. Comparação das características dos dados sísmicos

In [ ]:
with segyio.open(
    str(JEQUITINHONHA_PATH), "r", ignore_geometry=True
) as f:
    dt_jequitinhonha = f.bin[segyio.BinField.Interval] / 1000

with segyio.open(
    str(CAMPOS_PATH), "r", ignore_geometry=True
) as f:
    dt_campos = f.bin[segyio.BinField.Interval] / 1000


df_metadata = pd.DataFrame({
    "Jequitinhonha": {
        "Número de traços": seismic_data.shape[0],
        "Amostras por traço": seismic_data.shape[2],
        "Total de amostras": seismic_data.size,
        "Intervalo de amostragem (ms)": dt_jequitinhonha,
        "Memória utilizada (MiB)": round(
            seismic_data.nbytes / 1024**2, 2
        )
    },
    "Campos": {
        "Número de traços": seismic_data_campos.shape[0],
        "Amostras por traço": seismic_data_campos.shape[2],
        "Total de amostras": seismic_data_campos.size,
        "Intervalo de amostragem (ms)": dt_campos,
        "Memória utilizada (MiB)": round(
            seismic_data_campos.nbytes / 1024**2, 2
        )
    }
})

display(df_metadata)

### 4. Análise Exploratória dos Dados

A análise exploratória tem como objetivo investigar as características dos sinais sísmicos das bacias do Jequitinhonha e de Campos.

#### 4.1 Caracterização estatística

In [ ]:
def seismic_statistics(data):

    valid_mask = np.isfinite(data)
    valid_data = data[valid_mask]

    if valid_data.size == 0:
        raise ValueError("Nenhuma amostra válida.")

    return {
        "Amplitude mínima": np.min(valid_data),
        "Amplitude máxima": np.max(valid_data),
        "Média": np.mean(valid_data, dtype=np.float64),
        "Desvio padrão": np.std(
            valid_data, dtype=np.float64
        ),
        "Mediana": np.median(valid_data),
        "Percentil 99,5 absoluto": np.percentile(
            np.abs(valid_data), 99.5
        ),
        "Valores inválidos (%)": (
            100 * (1 - valid_mask.mean())
        ),
        "Valores zerados (%)": (
            100 * np.mean(valid_data == 0)
        )
    }


df_statistics = pd.DataFrame({
    "Jequitinhonha": seismic_statistics(seismic_data),
    "Campos": seismic_statistics(seismic_data_campos)
})

display(df_statistics.round(4))

#### 4.2. Visualização da amplitude sísmica

Os dados sísmicos são reorganizados em matrizes bidimensionais para visualização das amplitudes.

In [ ]:
# Conversão para matrizes bidimensionais
seismic_data_2d = seismic_data.reshape(
    -1, seismic_data.shape[2]
)

seismic_data_campos_2d = seismic_data_campos.reshape(
    -1, seismic_data_campos.shape[2]
)


def plot_seismic_amplitude(data, titulo):

    # Ajuste do contraste
    v_limit = np.nanpercentile(np.abs(data), 99.5)
    v_limit = v_limit if v_limit > 0 else 1.0

    # Redução de resolução apenas para visualização
    step = max(1, int(np.ceil(data.shape[0] / 1200)))

    data_display = data[::step]

    fig, ax = plt.subplots(figsize=(12, 6))

    im = ax.imshow(
        data_display.T,
        aspect="auto",
        cmap="seismic",
        vmin=-v_limit,
        vmax=v_limit,
        extent=[
            0,
            data.shape[0],
            data.shape[1],
            0
        ]
    )

    ax.set_title(titulo)
    ax.set_xlabel("Índice do traço")
    ax.set_ylabel("Amostras temporais")

    plt.colorbar(
        im,
        ax=ax,
        orientation="horizontal",
        fraction=0.046,
        pad=0.08,
        label="Amplitude sísmica"
    )

    plt.tight_layout()
    plt.show()


plot_seismic_amplitude(
    seismic_data_2d,
    "Amplitude Sísmica — Jequitinhonha"
)

plot_seismic_amplitude(
    seismic_data_campos_2d,
    "Amplitude Sísmica — Campos"
)

#### 4.3. Distribuição das amplitudes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

datasets = {
    "Jequitinhonha": seismic_data,
    "Campos": seismic_data_campos
}

for ax, (name, data) in zip(axes, datasets.items()):

    values = data.ravel()
    values = values[np.isfinite(values)]

    local_rng = np.random.default_rng(RANDOM_STATE)

    indices = local_rng.choice(
        len(values),
        size=min(100_000, len(values)),
        replace=False
    )

    sample = values[indices]

    ax.hist(
        sample,
        bins=100,
        color="steelblue",
        density=True,
        edgecolor="black",
        alpha=0.8
    )

    ax.set_title(name)
    ax.set_xlabel("Amplitude sísmica")
    ax.set_ylabel("Densidade")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

### 4.4. Análise espectral dos sinais

O conteúdo de frequência dos sinais sísmicos é investigado utilizando o método de Welch.

In [ ]:
from scipy.signal import welch


def mean_frequency_spectrum(data, dt_ms):

    data = data.reshape(-1, data.shape[-1])

    # Seleção de traços distribuídos pelo arquivo
    indices = np.linspace(
        0,
        len(data) - 1,
        min(256, len(data)),
        dtype=int
    )

    sample = data[indices]

    valid = (
        np.isfinite(sample).all(axis=1)
        & np.any(sample != 0, axis=1)
    )

    sample = sample[valid]

    fs = 1000 / dt_ms

    frequencies, power = welch(
        sample,
        fs=fs,
        axis=1,
        nperseg=min(256, sample.shape[1])
    )

    mean_power = power.mean(axis=0)

    return frequencies, mean_power / max(
        mean_power.max(), 1e-12
    )


freq_j, power_j = mean_frequency_spectrum(
    seismic_data, dt_jequitinhonha
)

freq_c, power_c = mean_frequency_spectrum(
    seismic_data_campos, dt_campos
)

plt.figure(figsize=(10, 5))

plt.plot(freq_j, power_j, label="Jequitinhonha")
plt.plot(freq_c, power_c, label="Campos")

plt.title("Comparação dos Espectros Médios")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Potência normalizada")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5. Extração de Atributos Sísmicos

A engenharia de atributos consiste na transformação das amplitudes sísmicas em um conjunto de características utilizadas como entrada dos algoritmos ML.

#### 5.1. Transformada de Hilbert e sinal analítico

O sinal analítico é obtido pela combinação do sinal
sísmico original com sua transformada de Hilbert:

$$
S_a(t)=S(t)+jH[S(t)]
$$

#### Envelope

$$
E(t)=|S_a(t)|
$$

#### Fase instantânea

$$
\phi(t)=\operatorname{atan2}
\left(
\operatorname{Im}(S_a(t)),
\operatorname{Re}(S_a(t))
\right)
$$

#### Frequência instantânea

$$
f(t)=\frac{1}{2\pi}\frac{d\phi(t)}{dt}
$$

#### Amplitude RMS

$$
RMS=\sqrt{\frac{1}{N}\sum_{i=1}^{N}S_i^2}
$$

#### Variância local

$$
\sigma^2=\frac{1}{N}\sum_{i=1}^{N}(S_i-\mu)^2
$$

Os atributos são calculados ao longo do eixo temporal
de cada traço sísmico.

#### 5.2. Implementação da extração de atributos

A transformada de Hilbert é aplicada ao eixo temporal
dos dados sísmicos, preservando a estrutura matricial
utilizada na metodologia original.

A extração é realizada inicialmente em blocos reduzidos
de traços, visando limitar o consumo de memória.

In [ ]:
import importlib
import src.features

importlib.reload(src.features)

from src.features import (
    extract_seismic_attributes,
    create_feature_matrix
)

print("Funções importadas com sucesso!")

In [ ]:
from src.features import (
    extract_seismic_attributes,
    create_feature_matrix
)

# Selecionar blocos centrais para visualização
start_j = max(0, seismic_data.shape[0] // 2 - 200)
start_c = max(0, seismic_data_campos.shape[0] // 2 - 200)

sample_jequitinhonha = seismic_data[start_j:start_j + 400]
sample_campos = seismic_data_campos[start_c:start_c + 400]

# Extração dos atributos
features_jequitinhonha = extract_seismic_attributes(
    sample_jequitinhonha,
    dt_jequitinhonha
)

features_campos = extract_seismic_attributes(
    sample_campos,
    dt_campos
)

print("Atributos extraídos com sucesso!")

for name, values in features_jequitinhonha.items():
    print(name, values.shape)

#### 5.3. Visualização dos atributos sísmicos

In [ ]:
def plot_seismic_attributes(features_list, basin_name):

    fig, axes = plt.subplots(
        2, 3, figsize=(14, 8)
    )

    color_maps = {
        "Amplitude": "seismic",
        "Envelope": "inferno",
        "Fase": "twilight",
        "Frequência": "coolwarm",
        "RMS": "viridis",
        "Variância": "magma"
    }

    for ax, (name, values) in zip(
        axes.flat,
        features_list.items()
    ):

        data = values.reshape(-1, values.shape[-1])

        if name == "Fase":

            vmin, vmax = -np.pi, np.pi

        elif name in ["Amplitude", "Frequência"]:

            limit = np.percentile(
                np.abs(data), 99.5
            ) or 1.0

            vmin, vmax = -limit, limit

        else:

            vmin = 0
            vmax = np.percentile(data, 99.5) or 1.0

        im = ax.imshow(
            data.T,
            aspect="auto",
            cmap=color_maps[name],
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(name)
        ax.set_xlabel("Traços")
        ax.set_ylabel("Amostras")

        fig.colorbar(im, ax=ax, fraction=0.046)

    fig.suptitle(
        f"Atributos Sísmicos — {basin_name}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()


plot_seismic_attributes(
    features_jequitinhonha,
    "Jequitinhonha"
)

plot_seismic_attributes(
    features_campos,
    "Campos"
)

### 5.4. Normalização das amplitudes sísmicas

A análise dos atributos revelou diferenças significativas
entre as escalas de amplitude dos dados das bacias do
Jequitinhonha e de Campos.

Para reduzir a influência dessas diferenças sobre os
modelos de Machine Learning, é aplicada uma normalização
robusta das amplitudes utilizando o percentil 99,5%
da amplitude absoluta de cada traço sísmico.

Após a normalização, os atributos são recalculados,
permitindo uma comparação baseada nas características
relativas dos sinais.

In [ ]:
# Atributos normalizados do Jequitinhonha
features_jequitinhonha_normalized = extract_seismic_attributes(
    sample_jequitinhonha,
    dt_jequitinhonha,
    normalize=True
)

# Atributos normalizados de Campos
features_campos_normalized = extract_seismic_attributes(
    sample_campos,
    dt_campos,
    normalize=True
)

print("Atributos normalizados extraídos com sucesso!")

In [ ]:
plot_seismic_attributes(
    features_jequitinhonha_normalized,
    "Jequitinhonha — Normalizado"
)

plot_seismic_attributes(
    features_campos_normalized,
    "Campos — Normalizado"
)

### 6. Separação e Pré-processamento dos Dados

Para a realização do treinamento, se faz necessário uma separação prévia dos dados em conjuntos de treinamento, ajuste de
hiperparâmetros e validação interna.

Os dados da Bacia de Campos são reservados para a avaliação da capacidade de generalização dos modelos.

### 6.1. Separação dos conjuntos de dados

A separação dos dados é realizada por blocos contíguos
de traços sísmicos da Bacia do Jequitinhonha.

Os dados são distribuídos da seguinte maneira:

- 64% para treinamento;
- 16% para ajuste de hiperparâmetros;
- 20% para validação interna.

In [ ]:
# Número total de traços
n_traces = seismic_data.shape[0]
n_traces_campos = seismic_data_campos.shape[0]

# Verificar as proporções
assert np.isclose(
    TRAIN_SIZE + TUNING_SIZE + VALIDATION_SIZE,
    1.0
)

assert min(
    TRAIN_SIZE,
    TUNING_SIZE,
    VALIDATION_SIZE
) > 0

# Limites dos conjuntos
cut_train = int(n_traces * TRAIN_SIZE)

cut_validation = int(
    n_traces * (TRAIN_SIZE + TUNING_SIZE)
)

# Índices dos traços de treinamento
train_indices = np.arange(0, cut_train)

# Índices para ajuste de hiperparâmetros
tuning_indices = np.arange(
    cut_train,
    cut_validation
)

# Índices para validação interna
validation_indices = np.arange(
    cut_validation,
    n_traces
)

# Índices da Bacia de Campos
campos_indices = np.arange(n_traces_campos)

# Resultados
print("Total de traços do Jequitinhonha:", n_traces)

print("Treinamento:", len(train_indices))
print("Ajuste:", len(tuning_indices))
print("Validação:", len(validation_indices))

print("\nTotal de traços de Campos:", n_traces_campos)
print("Teste externo:", len(campos_indices))

In [ ]:
# Verificar se os conjuntos não estão vazios
assert len(train_indices) > 0
assert len(tuning_indices) > 0
assert len(validation_indices) > 0

# Verificar sobreposição de traços
assert np.intersect1d(
    train_indices,
    tuning_indices
).size == 0

assert np.intersect1d(
    train_indices,
    validation_indices
).size == 0

assert np.intersect1d(
    tuning_indices,
    validation_indices
).size == 0

# Verificar se todos os traços foram considerados
all_indices = np.concatenate([
    train_indices,
    tuning_indices,
    validation_indices
])

assert np.array_equal(
    np.sort(all_indices),
    np.arange(n_traces)
)

print("Separação validada com sucesso!")
print("Nenhum traço compartilhado entre os conjuntos.")
print("Todos os traços foram considerados.")

In [ ]:
df_split = pd.DataFrame({

    "Conjunto": [
        "Treinamento",
        "Ajuste de hiperparâmetros",
        "Validação interna",
        "Teste externo"
    ],

    "Bacia": [
        "Jequitinhonha",
        "Jequitinhonha",
        "Jequitinhonha",
        "Campos"
    ],

    "Número de traços": [
        len(train_indices),
        len(tuning_indices),
        len(validation_indices),
        len(campos_indices)
    ],

    "Percentual (%)": [
        100 * len(train_indices) / n_traces,
        100 * len(tuning_indices) / n_traces,
        100 * len(validation_indices) / n_traces,
        100.0
    ]
})

display(df_split.round(2))

#### 6.2. Construção da matriz de atributos sísmicos

Os atributos sísmicos são organizados em uma matriz
bidimensional, na qual cada linha representa uma amostra
sísmica e cada coluna representa uma característica.

In [ ]:
# Atributos da metodologia original
original_feature_names = [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Atributos da metodologia expandida
feature_names = [
    "Amplitude",
    "Envelope",
    "Seno da fase",
    "Cosseno da fase",
    "Frequência",
    "RMS",
    "Variância"
]

# Todos os atributos disponíveis
all_feature_names = feature_names + ["Fase"]

print("Atributos da metodologia original:")
print(original_feature_names)

print("\nAtributos da metodologia expandida:")
print(feature_names)

print("\nTotal de colunas armazenadas:")
print(len(all_feature_names))

In [ ]:
from src.features import (
    extract_seismic_attributes,
    create_feature_matrix,
    normalize_seismic_traces
)

# Pequeno bloco para verificação
test_block = seismic_data[:10].copy()

# Extração sem normalização
test_original = extract_seismic_attributes(
    test_block,
    dt_jequitinhonha,
    normalize=False
)

# Extração com normalização
test_normalized = extract_seismic_attributes(
    test_block,
    dt_jequitinhonha,
    normalize=True
)

# Normalização de referência
expected_normalized = normalize_seismic_traces(
    test_block
)

# Verificar preservação da amplitude original
assert np.array_equal(
    test_original["Amplitude"],
    test_block
)

# Verificar aplicação da normalização
assert np.allclose(
    test_normalized["Amplitude"],
    expected_normalized
)

# Verificar preservação dos dados de entrada
assert np.array_equal(
    test_block,
    seismic_data[:10]
)

print("Normalização verificada com sucesso!")
print("Dados originais preservados.")

#### 6.4. Construção das amostras


In [ ]:
def build_sample(
    data,
    trace_indices,
    dt_ms,
    max_samples,
    seed=42
):

    # Gerador de números aleatórios
    local_rng = np.random.default_rng(seed)

    # Verificar formato dos dados
    if data.ndim != 3 or data.shape[1] != 1:
        raise ValueError(
            "Formato esperado: (n_traces, 1, n_samples)."
        )

    trace_indices = np.asarray(
        trace_indices,
        dtype=np.int64
    )

    n_samples = data.shape[2]

    # Número total de amostras disponíveis
    n_total = len(trace_indices) * n_samples

    if n_total == 0 or max_samples <= 0:
        raise ValueError("Conjunto de dados vazio.")

    # Seleção aleatória sem reposição
    selected = np.sort(
        local_rng.choice(
            n_total,
            size=min(max_samples, n_total),
            replace=False
        )
    )

    # Identificação dos traços e posições temporais
    selected_traces = selected // n_samples
    selected_times = selected % n_samples

    parts = []

    # Processamento em blocos de traços
    for start in range(
        0,
        len(trace_indices),
        TRACE_BATCH_SIZE
    ):

        stop = min(
            start + TRACE_BATCH_SIZE,
            len(trace_indices)
        )

        # Identificar as amostras pertencentes ao bloco
        mask = (
            (selected_traces >= start)
            & (selected_traces < stop)
        )

        if not mask.any():
            continue

        # Selecionar os dados sísmicos originais
        block = data[trace_indices[start:stop]]

        # Extrair atributos COM NORMALIZAÇÃO
        features = extract_seismic_attributes(
            block,
            dt_ms,
            normalize=True
        )

        # Índices relativos ao bloco
        row_indices = selected_traces[mask] - start
        time_indices = selected_times[mask]

        # Selecionar as amostras dos atributos
        sampled_features = {
            name: values[row_indices, 0, time_indices]
            for name, values in features.items()
        }

        # Empilhamento e construção da matriz
        matrix = create_feature_matrix(
            sampled_features,
            all_feature_names
        )

        # Verificar validade das amostras
        valid = (
            np.isfinite(matrix).all(axis=1)
            & (sampled_features["Envelope"] > 0)
        )

        # Armazenar as amostras válidas
        parts.append(matrix[valid])

    if not parts:
        raise ValueError("Nenhuma amostra disponível.")

    # Unir todas as matrizes
    X = np.concatenate(parts, axis=0)

    if len(X) == 0:
        raise ValueError("Nenhuma amostra válida.")

    return X.astype(np.float32, copy=False)

In [ ]:
# Conjunto de treinamento
X_train = build_sample(
    seismic_data,
    train_indices,
    dt_jequitinhonha,
    MAX_TRAIN_SAMPLES,
    seed=RANDOM_STATE
)

print("Treinamento preparado!")

# Conjunto de ajuste de hiperparâmetros
X_tuning = build_sample(
    seismic_data,
    tuning_indices,
    dt_jequitinhonha,
    MAX_VALIDATION_SAMPLES,
    seed=RANDOM_STATE + 1
)

print("Ajuste preparado!")

# Conjunto de validação interna
X_val = build_sample(
    seismic_data,
    validation_indices,
    dt_jequitinhonha,
    MAX_VALIDATION_SAMPLES,
    seed=RANDOM_STATE + 2
)

print("Validação preparada!")

# Dimensões das matrizes
print("\nDimensões das matrizes:")

print("X_train:", X_train.shape)
print("X_tuning:", X_tuning.shape)
print("X_val:", X_val.shape)

#### 6.5. Análise e verificação das matrizes

As matrizes de atributos são analisadas quanto às suas
dimensões, características estatísticas e presença de
valores inválidos.

In [ ]:
df_train = pd.DataFrame(
    X_train,
    columns=all_feature_names
)

display(df_train.head(10))

In [ ]:
df_attributes_statistics = df_train.describe().T

display(df_attributes_statistics.round(4))

In [ ]:
amplitude_index = all_feature_names.index(
    "Amplitude"
)

datasets = [
    ("Treinamento", X_train),
    ("Ajuste", X_tuning),
    ("Validação", X_val)
]

for name, matrix in datasets:

    # Verificar número de colunas
    assert matrix.shape[1] == len(all_feature_names)

    # Verificar existência de valores inválidos
    assert np.isfinite(matrix).all()

    amplitude = matrix[:, amplitude_index]

    print(f"\nConjunto: {name}")

    print("Dimensões:", matrix.shape)

    print(
        "Percentil 99,5% da amplitude:",
        np.percentile(np.abs(amplitude), 99.5)
    )

    print(
        "Amplitude máxima:",
        np.max(np.abs(amplitude))
    )

    print(
        "Valores finitos:",
        np.isfinite(matrix).all()
    )

print("\nTodas as matrizes foram verificadas!")

#### 6.6. Padronização dos atributos sísmicos

Os atributos sísmicos apresentam diferentes distribuições
e escalas numéricas.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Identificação das colunas da metodologia expandida
feature_indices = [
    all_feature_names.index(name)
    for name in feature_names
]

# Inicialização do padronizador
scaler = StandardScaler()

# Ajustar exclusivamente com o treinamento
X_train_scaled = scaler.fit_transform(
    X_train[:, feature_indices]
)

# Aplicar os mesmos parâmetros ao ajuste
X_tuning_scaled = scaler.transform(
    X_tuning[:, feature_indices]
)

# Aplicar os mesmos parâmetros à validação
X_val_scaled = scaler.transform(
    X_val[:, feature_indices]
)

print("Padronização concluída!")

print("\nDimensões após a padronização:")

print("Treinamento:", X_train_scaled.shape)
print("Ajuste:", X_tuning_scaled.shape)
print("Validação:", X_val_scaled.shape)

In [ ]:
df_scaled_statistics = pd.DataFrame({

    "Atributo": feature_names,

    "Média antes": X_train[
        :, feature_indices
    ].mean(axis=0),

    "Desvio antes": X_train[
        :, feature_indices
    ].std(axis=0),

    "Média depois": X_train_scaled.mean(axis=0),

    "Desvio depois": X_train_scaled.std(axis=0)
})

display(df_scaled_statistics.round(4))

In [ ]:
# Média e desvio padrão do treinamento
scaled_mean = X_train_scaled.mean(axis=0)

scaled_std = X_train_scaled.std(axis=0)

# Identificar atributos não constantes
non_constant = scaler.var_ > 0

# Verificar média aproximadamente zero
assert np.allclose(
    scaled_mean,
    0,
    atol=1e-3
)

# Verificar desvio aproximadamente unitário
assert np.allclose(
    scaled_std[non_constant],
    1,
    atol=1e-3
)

# Verificar valores finitos
assert np.isfinite(X_train_scaled).all()
assert np.isfinite(X_tuning_scaled).all()
assert np.isfinite(X_val_scaled).all()

print("Padronização validada com sucesso!")

print(
    "Atributos constantes:",
    np.array(feature_names)[~non_constant]
)

#### 6.7. Visualização dos efeitos da padronização


In [ ]:
# Gerador para seleção das amostras
local_rng = np.random.default_rng(
    RANDOM_STATE
)

# Selecionar amostras para visualização
plot_indices = local_rng.choice(
    len(X_train),
    size=min(5000, len(X_train)),
    replace=False
)

# Antes da padronização
data_before = X_train[
    plot_indices
][:, feature_indices]

# Depois da padronização
data_after = X_train_scaled[
    plot_indices
]

# Criação dos gráficos
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 5)
)

# Boxplot antes da padronização
axes[0].boxplot(
    data_before,
    tick_labels=feature_names,
    showfliers=False
)

axes[0].set_title(
    "Antes da padronização"
)

axes[0].set_ylabel(
    "Valor do atributo"
)

# Boxplot depois da padronização
axes[1].boxplot(
    data_after,
    tick_labels=feature_names,
    showfliers=False
)

axes[1].set_title(
    "Depois da padronização"
)

axes[1].set_ylabel(
    "Valor padronizado"
)

# Configurações dos gráficos
for ax in axes:

    ax.tick_params(
        axis="x",
        rotation=45
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.tight_layout()
plt.show()

### 7. Seleção de Atributos

Os atributos utilizados no treinamento dos modelos são:

- Amplitude sísmica;
- Envelope instantâneo;
- Fase instantânea.

In [ ]:
original_feature_names = [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Atributos utilizados pelos modelos
selected_features = original_feature_names.copy()

# Identificação das colunas
selected_indices = [
    all_feature_names.index(name)
    for name in selected_features
]

# Nomes das dimensões definitivas
final_feature_names = selected_features.copy()

print("Atributos selecionados:")

for name, index in zip(
    selected_features,
    selected_indices
):
    print(f"{name} — Coluna {index}")

print(
    "\nNúmero de atributos:",
    len(selected_features)
)

#### 7.2. Construção das matrizes 

As matrizes de atributos são construídas utilizando a amplitude, o envelope e a fase instantânea.

Cada linha representa uma amostra sísmica e cada coluna representa um dos três atributos selecionados.


In [ ]:
# Matriz de treinamento
X_train_original = X_train[:, selected_indices]

# Matriz de ajuste
X_tuning_original = X_tuning[:, selected_indices]

# Matriz de validação interna
X_val_original = X_val[:, selected_indices]

print("Dimensões das matrizes:")

print("Treinamento:", X_train_original.shape)

print("Ajuste:", X_tuning_original.shape)

print("Validação:", X_val_original.shape)

In [ ]:
df_selected_features = pd.DataFrame(

    X_train_original,

    columns=selected_features
)

display(
    df_selected_features.head(10)
)

In [ ]:
df_selected_statistics = (
    df_selected_features.describe().T
)

display(
    df_selected_statistics.round(4)
)

#### 7.3. Padronização dos atributos sísmicos

O StandardScaler é utilizado para padronizar os três
atributos sísmicos selecionados.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Padronizador específico da metodologia original
scaler_original = StandardScaler()

# Treinamento
X_train_final = scaler_original.fit_transform(
    X_train_original
)

# Ajuste de hiperparâmetros
X_tuning_final = scaler_original.transform(
    X_tuning_original
)

# Validação interna
X_val_final = scaler_original.transform(
    X_val_original
)

# Pré-processador utilizado pelos modelos
final_preprocessor = scaler_original

# Identificação da representação
best_experiment = "B"

print("Padronização concluída!")

print("\nDimensões das matrizes definitivas:")

print("Treinamento:", X_train_final.shape)

print("Ajuste:", X_tuning_final.shape)

print("Validação:", X_val_final.shape)

In [ ]:
df_final_statistics = pd.DataFrame({

    "Atributo": selected_features,

    "Média antes":
        X_train_original.mean(axis=0),

    "Desvio antes":
        X_train_original.std(axis=0),

    "Média depois":
        X_train_final.mean(axis=0),

    "Desvio depois":
        X_train_final.std(axis=0)
})

display(
    df_final_statistics.round(4)
)

#### 7.4. Visualização dos atributos sísmicos

In [ ]:
visual_rng = np.random.default_rng(
    RANDOM_STATE
)

visual_indices = visual_rng.choice(

    len(X_train_original),

    size=min(
        5000,
        len(X_train_original)
    ),

    replace=False
)

# Atributos antes da padronização
X_visual_original = X_train_original[
    visual_indices
]

# Atributos depois da padronização
X_visual_scaled = X_train_final[
    visual_indices
]

print(
    "Amostras utilizadas na visualização:",
    len(visual_indices)
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5)
)

# Antes da padronização
axes[0].boxplot(

    X_visual_original,

    tick_labels=selected_features,

    showfliers=False
)

axes[0].set_title(
    "Antes da padronização"
)

axes[0].set_ylabel(
    "Valor do atributo"
)

# Depois da padronização
axes[1].boxplot(

    X_visual_scaled,

    tick_labels=selected_features,

    showfliers=False
)

axes[1].set_title(
    "Depois da padronização"
)

axes[1].set_ylabel(
    "Valor padronizado"
)

for ax in axes:

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(
    figsize=(11, 8)
)

ax = fig.add_subplot(
    111,
    projection="3d"
)

ax.scatter(

    X_visual_original[:, 0],

    X_visual_original[:, 1],

    X_visual_original[:, 2],

    s=5,

    alpha=0.5,

    color="steelblue"
)

ax.set_xlabel(
    "Amplitude normalizada"
)

ax.set_ylabel(
    "Envelope"
)

ax.set_zlabel(
    "Fase instantânea (rad)"
)

ax.set_title(
    "Distribuição 3D dos Atributos Sísmicos"
)

plt.tight_layout()
plt.show()

In [ ]:
feature_pairs = [

    ("Amplitude", "Envelope"),

    ("Amplitude", "Fase"),

    ("Envelope", "Fase")
]

fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

for ax, (feature_x, feature_y) in zip(
    axes,
    feature_pairs
):

    idx_x = selected_features.index(
        feature_x
    )

    idx_y = selected_features.index(
        feature_y
    )

    ax.scatter(

        X_visual_original[:, idx_x],

        X_visual_original[:, idx_y],

        s=5,

        alpha=0.4,

        color="steelblue"
    )

    ax.set_xlabel(feature_x)

    ax.set_ylabel(feature_y)

    ax.set_title(
        f"{feature_x} × {feature_y}"
    )

    ax.grid(alpha=0.3)

plt.suptitle(
    "Relações entre os Atributos Sísmicos"
)

plt.tight_layout()
plt.show()

#### 7.5. Verificação final das matrizes

In [ ]:
# Verificação da quantidade de atributos
assert X_train_final.shape[1] == 3

assert X_tuning_final.shape[1] == 3

assert X_val_final.shape[1] == 3

# Verificação da quantidade de amostras
assert len(X_train_final) == len(X_train)

assert len(X_tuning_final) == len(X_tuning)

assert len(X_val_final) == len(X_val)

# Verificação de valores finitos
assert np.isfinite(X_train_final).all()

assert np.isfinite(X_tuning_final).all()

assert np.isfinite(X_val_final).all()

# Verificação dos atributos não constantes
assert np.all(scaler_original.var_ > 0)

# Verificação da média
assert np.allclose(

    X_train_final.mean(axis=0),

    0,

    atol=1e-3
)

# Verificação do desvio padrão
assert np.allclose(

    X_train_final.std(axis=0),

    1,

    atol=1e-3
)

# Verificar preservação da etapa 6
assert X_train_scaled.shape[1] == 7

assert scaler.n_features_in_ == 7

assert scaler_original.n_features_in_ == 3

assert scaler_original is not scaler

print("Todas as verificações foram concluídas!")

print("\nAtributos definitivos:")
print(selected_features)

print("\nMatrizes definitivas:")
print("Treinamento:", X_train_final.shape)
print("Ajuste:", X_tuning_final.shape)
print("Validação:", X_val_final.shape)


print("Dados preparados para o K-means e GMM!")

### 8. Agrupamento utilizando K-means

O algoritmo K-means é utilizado para realizar o agrupamento
não supervisionado das amostras sísmicas, considerando
as características extraídas na etapa de engenharia de
atributos.

A função objetivo é definida por:

$$
J = \sum_{j=1}^{K}\sum_{x_i \in C_j}
\|x_i-\mu_j\|^2
$$

#### 8.1. Preparação dos dados para o K-means

In [ ]:
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score
)

from matplotlib.colors import (
    BoundaryNorm,
    ListedColormap
)

import time

# Verificar os atributos selecionados
assert selected_features == [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Verificar as matrizes definitivas
assert X_train_final.shape[1] == 3
assert X_tuning_final.shape[1] == 3
assert X_val_final.shape[1] == 3

# Verificar os padronizadores
assert scaler.n_features_in_ == 7
assert scaler_original.n_features_in_ == 3

assert final_preprocessor is scaler_original

# Verificar valores inválidos
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_tuning_final).all()
assert np.isfinite(X_val_final).all()

print("Dados verificados com sucesso!")

print("\nAtributos utilizados:")
print(selected_features)

print("\nDimensões:")

print("Treinamento:", X_train_final.shape)
print("Ajuste:", X_tuning_final.shape)
print("Validação:", X_val_final.shape)

print("\nPré-processador:")
print(final_preprocessor)

In [ ]:
# Intervalo de valores de K
K_range = list(K_RANGE)

# Amostras destinadas ao treinamento
X_sample_base = X_train_final

# Gerador aleatório
kmeans_rng = np.random.default_rng(
    RANDOM_STATE
)

# Selecionar amostras para avaliação
kmeans_evaluation_indices = kmeans_rng.choice(

    len(X_tuning_final),

    size=min(
        SILHOUETTE_SAMPLES,
        len(X_tuning_final)
    ),

    replace=False
)

# Amostra de avaliação
X_kmeans_evaluation = X_tuning_final[
    kmeans_evaluation_indices
]

print(
    "Amostras de treinamento:",
    X_sample_base.shape
)

print(
    "Amostras de avaliação:",
    X_kmeans_evaluation.shape
)

print(
    "Valores de K:",
    K_range
)

#### 8.3. Avaliação do número de clusters

São avaliados diferentes valores de K, variando entre
2 e 9 clusters.

Para cada configuração, o algoritmo é ajustado sobre
as amostras de treinamento.

In [ ]:
# Armazenamento dos resultados
inertia_scores = []
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []
training_times = []

kmeans_results = []

# Armazenar os modelos treinados
kmeans_models = {}

# Armazenar os rótulos de avaliação
kmeans_evaluation_labels = {}

print("--- Avaliação do número ideal de clusters ---")

for k in K_range:

    # Configuração do algoritmo
    kmeans_model = KMeans(

        n_clusters=k,

        init="k-means++",

        n_init=10,

        max_iter=300,

        random_state=RANDOM_STATE
    )

    # Treinamento
    start_time = time.perf_counter()

    kmeans_model.fit(
        X_sample_base
    )

    training_time = (
        time.perf_counter() - start_time
    )

    # Predição no conjunto de ajuste
    labels_sample = kmeans_model.predict(
        X_kmeans_evaluation
    )

    # Número de clusters encontrados
    n_labels = len(
        np.unique(labels_sample)
    )

    # Inércia
    inertia = kmeans_model.inertia_

    # Cálculo das métricas
    if 2 <= n_labels < len(labels_sample):

        silhouette = silhouette_score(
            X_kmeans_evaluation,
            labels_sample
        )

        db = davies_bouldin_score(
            X_kmeans_evaluation,
            labels_sample
        )

        ch = calinski_harabasz_score(
            X_kmeans_evaluation,
            labels_sample
        )

    else:

        silhouette = np.nan
        db = np.nan
        ch = np.nan

    # Distribuição dos clusters
    counts = np.bincount(
        labels_sample,
        minlength=k
    )

    percentages = (
        100 * counts / len(labels_sample)
    )

    # Armazenamento das métricas
    inertia_scores.append(inertia)
    silhouette_scores.append(silhouette)
    davies_bouldin_scores.append(db)
    calinski_harabasz_scores.append(ch)
    training_times.append(training_time)

    # Armazenamento do modelo e dos rótulos
    kmeans_models[k] = kmeans_model

    kmeans_evaluation_labels[k] = labels_sample

    # Armazenamento dos resultados
    kmeans_results.append({

        "K": k,

        "Inércia": inertia,

        "Silhouette": silhouette,

        "Davies-Bouldin": db,

        "Calinski-Harabasz": ch,

        "Tempo (s)": training_time,

        "Clusters encontrados": n_labels,

        "Maior cluster (%)":
            percentages.max(),

        "Menor cluster (%)":
            percentages.min()
    })

    print(

        f"K={k} | "

        f"Silhouette={silhouette:.4f} | "

        f"DB={db:.4f} | "

        f"Maior cluster={percentages.max():.2f}% | "

        f"Tempo={training_time:.2f}s"
    )

print("\nAvaliação concluída!")

In [ ]:
df_kmeans_results = pd.DataFrame(
    kmeans_results
)

display(
    df_kmeans_results.round(4)
)

#### 8.4. Método do cotovelo

O método do cotovelo é utilizado para investigar
a relação entre o número de clusters e a inércia
do modelo K-means.

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(

    K_range,

    inertia_scores,

    marker="o",

    color="steelblue",

    linewidth=2
)

plt.title(
    "Método do Cotovelo — K-means"
)

plt.xlabel(
    "Número de clusters (K)"
)

plt.ylabel(
    "Inércia"
)

plt.xticks(K_range)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
df_inertia_reduction = pd.DataFrame({

    "K": K_range,

    "Inércia": inertia_scores
})

df_inertia_reduction[
    "Redução absoluta"
] = (
    -df_inertia_reduction["Inércia"].diff()
)

df_inertia_reduction[
    "Redução relativa (%)"
] = (

    -100

    * df_inertia_reduction[
        "Inércia"
    ].pct_change()
)

display(
    df_inertia_reduction.round(4)
)

#### 8.5. Método do coeficiente de Silhouette

O coeficiente de Silhouette é utilizado como principal
critério para a determinação do número de clusters.

A métrica considera a coesão interna dos agrupamentos
e a separação entre diferentes clusters.

A configuração associada ao maior valor de Silhouette
no conjunto de ajuste é identificada como candidata
principal para o treinamento definitivo.

Os índices Davies-Bouldin e Calinski-Harabasz são
utilizados como métricas complementares.

In [ ]:
# Verificar configurações válidas
valid_kmeans_results = df_kmeans_results[

    np.isfinite(
        df_kmeans_results["Silhouette"]
    )

    & (
        df_kmeans_results["Clusters encontrados"]
        == df_kmeans_results["K"]
    )
]

if valid_kmeans_results.empty:

    raise ValueError(
        "Nenhuma configuração válida encontrada."
    )

# Identificar maior Silhouette
best_row = valid_kmeans_results.loc[

    valid_kmeans_results["Silhouette"].idxmax()
]

# Número de clusters com maior Silhouette
best_k = int(best_row["K"])

# Melhor valor encontrado
best_silhouette = float(
    best_row["Silhouette"]
)

print(
    "Número de clusters com maior Silhouette:",
    best_k
)

print(
    "Melhor Silhouette:",
    round(best_silhouette, 4)
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(

    K_range,

    silhouette_scores,

    marker="o",

    color="steelblue",

    linewidth=2
)

# Destacar o maior valor
plt.scatter(

    best_k,

    best_silhouette,

    color="red",

    s=100,

    zorder=5,

    label=f"Maior Silhouette: K = {best_k}"
)

plt.axvline(

    best_k,

    color="red",

    linestyle="--",

    alpha=0.5
)

plt.title(
    "Coeficiente de Silhouette — K-means"
)

plt.xlabel(
    "Número de clusters (K)"
)

plt.ylabel(
    "Coeficiente de Silhouette"
)

plt.xticks(K_range)

plt.grid(alpha=0.3)

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

# Davies-Bouldin
axes[0].plot(

    K_range,

    davies_bouldin_scores,

    marker="o",

    color="darkorange"
)

axes[0].set_title(
    "Índice Davies-Bouldin"
)

axes[0].set_ylabel(
    "Davies-Bouldin"
)

# Calinski-Harabasz
axes[1].plot(

    K_range,

    calinski_harabasz_scores,

    marker="o",

    color="seagreen"
)

axes[1].set_title(
    "Índice Calinski-Harabasz"
)

axes[1].set_ylabel(
    "Calinski-Harabasz"
)

for ax in axes:

    ax.axvline(

        best_k,

        color="red",

        linestyle="--",

        alpha=0.5
    )

    ax.set_xlabel(
        "Número de clusters (K)"
    )

    ax.set_xticks(K_range)

    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### 8.6. Análise complementar dos agrupamentos

In [ ]:
cluster_distribution_results = []

for k in K_range:

    labels = kmeans_evaluation_labels[k]

    counts = np.bincount(
        labels,
        minlength=k
    )

    for cluster_id in range(k):

        cluster_distribution_results.append({

            "K": k,

            "Cluster": cluster_id,

            "Quantidade":
                counts[cluster_id],

            "Percentual (%)":
                100 * counts[cluster_id]
                / len(labels)
        })

df_all_cluster_distributions = pd.DataFrame(
    cluster_distribution_results
)

display(
    df_all_cluster_distributions.round(4)
)

In [ ]:
fig, axes = plt.subplots(

    2, 4,

    figsize=(18, 9),

    sharey=True
)

for ax, k in zip(
    axes.ravel(),
    K_range
):

    df_k = df_all_cluster_distributions[

        df_all_cluster_distributions["K"] == k
    ]

    colors_k = plt.get_cmap(
        "Spectral",
        k
    )(np.arange(k))

    bars = ax.bar(

        df_k["Cluster"],

        df_k["Percentual (%)"],

        color=colors_k
    )

    ax.bar_label(

        bars,

        fmt="%.1f",

        fontsize=8
    )

    ax.set_title(f"K = {k}")

    ax.set_xlabel("Cluster")

    ax.set_ylabel("Percentual (%)")

    ax.set_xticks(np.arange(k))

    ax.set_ylim(0, 110)

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.suptitle(
    "Distribuição dos Clusters para Diferentes Valores de K",
    fontsize=15
)

plt.tight_layout()
plt.show()

#### 8.7. Treinamento do K-means

In [ ]:
K_IDEAL = best_k

# Recuperar o modelo previamente treinado
final_kmeans = kmeans_models[K_IDEAL]

# Rótulos do treinamento
labels_train_kmeans = final_kmeans.labels_

# Rótulos do conjunto de ajuste
labels_tuning_kmeans = final_kmeans.predict(

    X_tuning_final
)

# Recuperar tempo de treinamento
kmeans_training_time = float(

    df_kmeans_results.loc[

        df_kmeans_results["K"] == K_IDEAL,

        "Tempo (s)"

    ].iloc[0]
)

print("Modelo definitivo selecionado!")

print(
    "Número de clusters:",
    K_IDEAL
)

print(
    "Dimensões dos centroides:",
    final_kmeans.cluster_centers_.shape
)

print(
    "Inércia final:",
    final_kmeans.inertia_
)

print(
    "Número de iterações:",
    final_kmeans.n_iter_
)

print(
    f"Tempo de treinamento: "
    f"{kmeans_training_time:.2f} segundos"
)

#### 8.8. Caracterização estatística dos centroides

Os centroides representam as características médias
dos agrupamentos identificados pelo algoritmo K-means.

In [ ]:
# Centroides na escala padronizada
centroids_scaled = (
    final_kmeans.cluster_centers_
)

# Transformação inversa
centroids_original_scale = (

    scaler_original.inverse_transform(
        centroids_scaled
    )
)

# Tabela dos centroides padronizados
df_centroids_scaled = pd.DataFrame(

    centroids_scaled,

    columns=selected_features
)

df_centroids_scaled.index.name = "Cluster"

# Tabela dos centroides na escala anterior
df_centroids = pd.DataFrame(

    centroids_original_scale,

    columns=selected_features
)

df_centroids.index.name = "Cluster"

print("Centroides padronizados:")

display(
    df_centroids_scaled.round(4)
)

print("\nCentroides na escala dos atributos:")

display(
    df_centroids.round(4)
)

In [ ]:
df_cluster_attributes = pd.DataFrame(

    X_train_original,

    columns=selected_features
)

df_cluster_attributes["Cluster"] = (
    labels_train_kmeans
)

df_cluster_statistics = (

    df_cluster_attributes

    .groupby("Cluster")[selected_features]

    .agg(["mean", "std", "min", "max"])
)

display(
    df_cluster_statistics.round(4)
)

In [ ]:
# Mapa de cores dos clusters
cluster_colors = plt.get_cmap(

    "Spectral",

    K_IDEAL
)

colors = cluster_colors(
    np.arange(K_IDEAL)
)

# Gráfico de barras dos centroides
df_centroids_scaled.T.plot(

    kind="bar",

    figsize=(12, 6),

    color=colors
)

plt.title(

    f"Caracterização dos Centroides "
    f"K-means (K={K_IDEAL})"
)

plt.xlabel(
    "Atributo sísmico"
)

plt.ylabel(
    "Valor padronizado do centroide"
)

plt.axhline(

    0,

    color="black",

    linewidth=0.8
)

plt.xticks(

    rotation=0
)

plt.legend(
    title="Cluster"
)

plt.grid(

    axis="y",

    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# Médias empíricas das amostras de cada cluster
df_cluster_means = (

    df_cluster_attributes

    .groupby("Cluster")[selected_features]

    .mean()

    .reindex(range(K_IDEAL))
)

# Comparação com os centroides recuperados
df_centroid_difference = (

    df_centroids[selected_features]

    - df_cluster_means
).abs()

print(
    "Diferença entre centroides "
    "e médias empíricas:"
)

display(
    df_centroid_difference.round(6)
)

print(

    "Maior diferença encontrada:",

    df_centroid_difference.to_numpy().max()
)

#### 8.9. Distribuição percentual dos clusters

In [ ]:
# Contagem no treinamento
cluster_counts_train = np.bincount(

    labels_train_kmeans,

    minlength=K_IDEAL
)

# Contagem no ajuste
cluster_counts_tuning = np.bincount(

    labels_tuning_kmeans,

    minlength=K_IDEAL
)

# Construção da tabela
df_kmeans_distribution = pd.DataFrame({

    "Cluster": np.arange(K_IDEAL),

    "Treinamento":
        cluster_counts_train,

    "Treinamento (%)":
        100 * cluster_counts_train
        / len(labels_train_kmeans),

    "Ajuste":
        cluster_counts_tuning,

    "Ajuste (%)":
        100 * cluster_counts_tuning
        / len(labels_tuning_kmeans)
})

display(
    df_kmeans_distribution.round(4)
)

In [ ]:
fig, axes = plt.subplots(

    1, 2,

    figsize=(14, 5),

    sharey=True
)

# Treinamento
bars_train = axes[0].bar(

    np.arange(K_IDEAL),

    df_kmeans_distribution[
        "Treinamento (%)"
    ],

    color=colors
)

axes[0].set_title(
    "Distribuição — Treinamento"
)

# Ajuste
bars_tuning = axes[1].bar(

    np.arange(K_IDEAL),

    df_kmeans_distribution[
        "Ajuste (%)"
    ],

    color=colors
)

axes[1].set_title(
    "Distribuição — Ajuste"
)

axes[0].bar_label(

    bars_train,

    fmt="%.2f%%"
)

axes[1].bar_label(

    bars_tuning,

    fmt="%.2f%%"
)

for ax in axes:

    ax.set_xlabel("Cluster")

    ax.set_ylabel(
        "Percentual de amostras (%)"
    )

    ax.set_xticks(
        np.arange(K_IDEAL)
    )

    ax.set_ylim(0, 110)

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.tight_layout()
plt.show()

#### 8.10. Visualização dos agrupamentos

In [ ]:
visual_rng = np.random.default_rng(

    RANDOM_STATE
)

visual_indices = visual_rng.choice(

    len(X_train_original),

    size=min(
        10000,
        len(X_train_original)
    ),

    replace=False
)

# Atributos antes da padronização
X_attributes_visual = X_train_original[
    visual_indices
]

# Rótulos correspondentes
labels_kmeans_visual = labels_train_kmeans[
    visual_indices
]

# Normalização das cores dos clusters
cluster_norm = BoundaryNorm(

    np.arange(
        -0.5,
        K_IDEAL + 0.5,
        1
    ),

    K_IDEAL
)

print(
    "Amostras utilizadas:",
    len(visual_indices)
)

In [ ]:
fig = plt.figure(

    figsize=(12, 8)
)

ax = fig.add_subplot(

    111,

    projection="3d"
)

# Visualização dos agrupamentos
scatter = ax.scatter(

    X_attributes_visual[:, 0],

    X_attributes_visual[:, 1],

    X_attributes_visual[:, 2],

    c=labels_kmeans_visual,

    cmap=cluster_colors,

    norm=cluster_norm,

    s=6,

    alpha=0.5
)

# Visualização dos centroides
ax.scatter(

    centroids_original_scale[:, 0],

    centroids_original_scale[:, 1],

    centroids_original_scale[:, 2],

    marker="X",

    c="black",

    s=150,

    label="Centroides"
)

ax.set_xlabel(
    "Amplitude normalizada"
)

ax.set_ylabel(
    "Envelope"
)

ax.set_zlabel(
    "Fase instantânea (rad)"
)

ax.set_title(

    f"Agrupamento Tridimensional "
    f"K-means (K={K_IDEAL})"
)

ax.legend()

fig.colorbar(

    scatter,

    ax=ax,

    ticks=np.arange(K_IDEAL),

    label="Cluster",

    shrink=0.7
)

plt.tight_layout()
plt.show()

#### 8.11. Segmentação sísmica da Bacia do Jequitinhonha

O modelo K-means ajustado sobre as amostras de
treinamento é aplicado à linha sísmica completa
da Bacia do Jequitinhonha.

In [ ]:
def segment_seismic_data(

    data,

    dt_ms,

    model,

    preprocessor,

    selected_indices
):

    # Verificar formato dos dados
    if data.ndim != 3 or data.shape[1] != 1:

        raise ValueError(

            "Formato esperado: "
            "(n_traces, 1, n_samples)"
        )

    n_traces = data.shape[0]

    # Criar volume de segmentação
    segmentation = np.full(

        data.shape,

        -1,

        dtype=np.int16
    )

    # Processamento em blocos
    for start in range(

        0,

        n_traces,

        TRACE_BATCH_SIZE
    ):

        stop = min(

            start + TRACE_BATCH_SIZE,

            n_traces
        )

        # Selecionar bloco original
        block = data[start:stop]

        # Extrair atributos normalizados
        features = extract_seismic_attributes(

            block,

            dt_ms,

            normalize=True
        )

        # Construir matriz de atributos
        X_block = create_feature_matrix(

            features,

            all_feature_names
        )

        # Aplicar o mesmo filtro da etapa 6
        valid = (

            np.isfinite(X_block).all(axis=1)

            & (
                features["Envelope"].reshape(-1) > 0
            )
        )

        # Inicializar rótulos do bloco
        block_labels = np.full(

            len(X_block),

            -1,

            dtype=np.int16
        )

        if valid.any():

            # Selecionar amplitude, envelope e fase
            X_valid = X_block[

                valid

            ][:, selected_indices]

            # Padronizar com parâmetros do treinamento
            X_valid_scaled = (

                preprocessor.transform(
                    X_valid
                )
            )

            # Aplicar o modelo treinado
            block_labels[valid] = (

                model.predict(
                    X_valid_scaled
                ).astype(np.int16)
            )

        # Reconstruir formato original
        segmentation[start:stop] = (

            block_labels.reshape(
                block.shape
            )
        )

        # Mostrar progresso
        if (

            start % (TRACE_BATCH_SIZE * 10) == 0

            or stop == n_traces
        ):

            print(

                f"Traços processados: "

                f"{stop}/{n_traces}"
            )

    return segmentation

In [ ]:
print(

    f"Segmentação da Bacia do Jequitinhonha "

    f"com K={K_IDEAL}"
)

start_time = time.perf_counter()

# Aplicação do modelo
segmentation_volume = segment_seismic_data(

    seismic_data,

    dt_jequitinhonha,

    final_kmeans,

    final_preprocessor,

    selected_indices
)

# Tempo de processamento
kmeans_segmentation_time = (

    time.perf_counter() - start_time
)

# Preservar nomenclatura anterior
cluster_labels_flat = (

    segmentation_volume.reshape(-1)
)

# Verificar dimensões
assert (

    segmentation_volume.shape

    == seismic_data.shape
)

# Verificar validade dos rótulos
assert np.all(

    (cluster_labels_flat >= -1)

    & (cluster_labels_flat < K_IDEAL)
)

print("\nSegmentação concluída!")

print(

    "Dimensões originais:",

    seismic_data.shape
)

print(

    "Dimensões da segmentação:",

    segmentation_volume.shape
)

print(

    "Rótulos encontrados:",

    np.unique(cluster_labels_flat)
)

print(

    f"Tempo de segmentação: "

    f"{kmeans_segmentation_time:.2f} segundos"
)

In [ ]:
# Selecionar amostras classificadas
valid_cluster_labels = cluster_labels_flat[

    cluster_labels_flat >= 0
]

# Contagem por cluster
cluster_counts = np.bincount(

    valid_cluster_labels,

    minlength=K_IDEAL
)

# Quantidade de amostras
total_voxels = len(
    cluster_labels_flat
)

valid_voxels = len(
    valid_cluster_labels
)

invalid_voxels = (

    total_voxels - valid_voxels
)

if valid_voxels == 0:

    raise ValueError(

        "Nenhuma amostra foi classificada."
    )

# Acrescentar distribuição à tabela de centroides
df_centroids["Quantidade de amostras"] = (

    cluster_counts
)

df_centroids["% Volume Total"] = (

    100 * cluster_counts

    / total_voxels
)

df_centroids["% Volume Classificado"] = (

    100 * cluster_counts

    / valid_voxels
)

display(
    df_centroids.round(4)
)

print(

    "Total de amostras:",

    total_voxels
)

print(

    "Amostras classificadas:",

    valid_voxels
)

print(

    "Amostras não classificadas:",

    invalid_voxels
)

print(

    "Percentual classificado:",

    round(

        100 * valid_voxels / total_voxels,

        4
    )
)

In [ ]:
# Dados sísmicos em duas dimensões
seismic_data_2d = seismic_data.reshape(

    -1,

    seismic_data.shape[2]
)

# Segmentação em duas dimensões
segmentation_volume_2d = (

    segmentation_volume.reshape(

        -1,

        segmentation_volume.shape[2]
    )
)

# Mapa de cores
cmap_spectral = plt.get_cmap(

    "Spectral",

    K_IDEAL
).copy()

# Regiões não classificadas
cmap_spectral.set_bad(

    color="lightgray"
)

print(

    "Dados sísmicos:",

    seismic_data_2d.shape
)

print(

    "Segmentação:",

    segmentation_volume_2d.shape
)

In [ ]:
# Limite de contraste
v_limit = np.percentile(

    np.abs(seismic_data_2d),

    99.5
)

if v_limit <= 0:

    v_limit = 1.0

# Mascarar regiões não classificadas
segmentation_masked = np.ma.masked_equal(

    segmentation_volume_2d,

    -1
)

# Criação da figura
fig, axes = plt.subplots(

    1, 2,

    figsize=(18, 7),

    sharey=True
)

# Amplitude sísmica original
im1 = axes[0].imshow(

    seismic_data_2d.T,

    cmap="seismic",

    vmin=-v_limit,

    vmax=v_limit,

    aspect="auto",

    interpolation="nearest"
)

axes[0].set_title(

    "Amplitude Sísmica Original"
)

# Segmentação K-means
im2 = axes[1].imshow(

    segmentation_masked.T,

    cmap=cmap_spectral,

    norm=cluster_norm,

    aspect="auto",

    interpolation="nearest"
)

axes[1].set_title(

    f"Segmentação K-means (K={K_IDEAL})"
)

for ax in axes:

    ax.set_xlabel(

        "Índice do traço"
    )

axes[0].set_ylabel(

    "Amostra temporal"
)

fig.colorbar(

    im1,

    ax=axes[0],

    label="Amplitude",

    fraction=0.046
)

fig.colorbar(

    im2,

    ax=axes[1],

    ticks=np.arange(K_IDEAL),

    label="Cluster",

    fraction=0.046
)

plt.suptitle(

    "Segmentação Sísmica — Bacia do Jequitinhonha",

    fontsize=15
)

plt.tight_layout()
plt.show()

#### 8.12. Visualização individual dos clusters

In [ ]:
def plot_individual_clusters_optimized():

    n_clusters = K_IDEAL

    # Subamostragem para visualização
    sample_step_traces = 4
    sample_step_samples = 2

    seg_sampled = segmentation_volume_2d[

        ::sample_step_traces,

        ::sample_step_samples
    ]

    print(

        "Formato da segmentação amostrada:",

        seg_sampled.shape
    )

    # Organização dos gráficos
    n_cols = min(3, n_clusters)

    n_rows = (

        n_clusters + n_cols - 1

    ) // n_cols

    fig, axes = plt.subplots(

        n_rows,

        n_cols,

        figsize=(5.5 * n_cols, 4 * n_rows),

        squeeze=False
    )

    axes = axes.ravel()

    for cluster_id in range(n_clusters):

        # Máscara binária
        cluster_mask = (

            seg_sampled == cluster_id
        )

        # Cor correspondente ao cluster
        individual_cmap = ListedColormap([

            "white",

            colors[cluster_id]
        ])

        ax = axes[cluster_id]

        ax.imshow(

            cluster_mask.T,

            cmap=individual_cmap,

            vmin=0,

            vmax=1,

            aspect="auto",

            interpolation="nearest",

            extent=(

                -0.5 * sample_step_traces,

                (seg_sampled.shape[0] - 0.5)
                * sample_step_traces,

                (seg_sampled.shape[1] - 0.5)
                * sample_step_samples,

                -0.5 * sample_step_samples
            )
        )

        # Percentual do cluster
        percentage = df_centroids.loc[

            cluster_id,

            "% Volume Classificado"
        ]

        ax.set_title(

            f"Cluster {cluster_id} — "

            f"{percentage:.2f}%"
        )

        ax.set_xlabel(

            "Índice do traço"
        )

        ax.set_ylabel(

            "Amostra temporal"
        )

    # Remover gráficos excedentes
    for ax in axes[n_clusters:]:

        ax.axis("off")

    plt.suptitle(

        f"Visualização Individual dos Clusters "

        f"(K={K_IDEAL})",

        fontsize=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_individual_clusters_optimized()

### 9. Agrupamento utilizando Gaussian Mixture Model (GMM)

O Gaussian Mixture Model (GMM) é um modelo probabilístico
de agrupamento não supervisionado que representa a
distribuição dos dados por uma combinação de distribuições
gaussianas multivariadas.

A função de densidade de probabilidade é definida por:

$$
p(x)=\sum_{k=1}^{K}\pi_k
\mathcal{N}(x\mid\mu_k,\Sigma_k)
$$

#### 9.1. Preparação dos dados para o GMM

As matrizes de atributos utilizadas no GMM correspondem
às mesmas matrizes empregadas no treinamento do K-means.

In [ ]:
from sklearn.mixture import GaussianMixture

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

import time

# Verificação dos atributos
assert selected_features == [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Verificação das dimensões
assert X_train_final.shape[1] == 3
assert X_tuning_final.shape[1] == 3
assert X_val_final.shape[1] == 3

# Verificação do pré-processador
assert final_preprocessor is scaler_original
assert scaler_original.n_features_in_ == 3

# Verificação do K-means
assert final_kmeans.n_features_in_ == 3

# Verificação dos dados
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_tuning_final).all()

print("Dados preparados para o GMM!")

print("\nAtributos utilizados:")
print(selected_features)

print("\nTreinamento:", X_train_final.shape)
print("Ajuste:", X_tuning_final.shape)

print("\nNúmero de clusters do K-means:")
print(K_IDEAL)

In [ ]:
# Mesmo intervalo avaliado no K-means
GMM_range = list(K_RANGE)

# Mesmos dados de treinamento
X_gmm_train = X_train_final

# Mesmos índices utilizados na avaliação do K-means
gmm_evaluation_indices = (
    kmeans_evaluation_indices.copy()
)

# Mesmas amostras de avaliação
X_gmm_evaluation = X_tuning_final[
    gmm_evaluation_indices
]

# Garantir condições iguais de comparação
assert np.array_equal(
    X_gmm_evaluation,
    X_kmeans_evaluation
)

print("Treinamento:", X_gmm_train.shape)

print("Avaliação:", X_gmm_evaluation.shape)

print("Componentes avaliadas:", GMM_range)

#### 9.2. Avaliação do número de componentes gaussianas

São avaliados modelos GMM com diferentes números
de componentes, variando entre 2 e 9.

In [ ]:
# Armazenamento das métricas
aic_scores = []
bic_scores = []

gmm_silhouette_scores = []
gmm_davies_bouldin_scores = []
gmm_calinski_harabasz_scores = []
gmm_training_times = []

# Armazenamento dos modelos e resultados
gmm_models = {}
gmm_results = []
gmm_evaluation_labels = {}

print("--- Avaliação do GMM ---")

for k in GMM_range:

    # Configuração do modelo
    gmm_model = GaussianMixture(

        n_components=k,

        covariance_type="full",

        init_params="kmeans",

        n_init=5,

        max_iter=300,

        reg_covar=1e-6,

        tol=1e-3,

        random_state=RANDOM_STATE
    )

    # Treinamento
    start_time = time.perf_counter()

    gmm_model.fit(X_gmm_train)

    training_time = (
        time.perf_counter() - start_time
    )

    # Critérios de informação
    aic = gmm_model.aic(X_gmm_train)

    bic = gmm_model.bic(X_gmm_train)

    # Log-verossimilhança média no ajuste
    tuning_log_likelihood = gmm_model.score(
        X_tuning_final
    )

    # Predição das amostras de avaliação
    labels_gmm = gmm_model.predict(
        X_gmm_evaluation
    )

    # Número de agrupamentos encontrados
    n_labels = len(
        np.unique(labels_gmm)
    )

    # Métricas complementares
    if 2 <= n_labels < len(labels_gmm):

        silhouette = silhouette_score(
            X_gmm_evaluation,
            labels_gmm
        )

        db = davies_bouldin_score(
            X_gmm_evaluation,
            labels_gmm
        )

        ch = calinski_harabasz_score(
            X_gmm_evaluation,
            labels_gmm
        )

    else:

        silhouette = np.nan
        db = np.nan
        ch = np.nan

    # Distribuição dos agrupamentos
    counts = np.bincount(
        labels_gmm,
        minlength=k
    )

    percentages = (
        100 * counts / len(labels_gmm)
    )

    # Armazenar resultados
    aic_scores.append(aic)
    bic_scores.append(bic)

    gmm_silhouette_scores.append(silhouette)

    gmm_davies_bouldin_scores.append(db)

    gmm_calinski_harabasz_scores.append(ch)

    gmm_training_times.append(training_time)

    # Armazenar modelo e rótulos
    gmm_models[k] = gmm_model

    gmm_evaluation_labels[k] = labels_gmm

    # Tabela de resultados
    gmm_results.append({

        "K": k,

        "AIC": aic,

        "BIC": bic,

        "Silhouette": silhouette,

        "Davies-Bouldin": db,

        "Calinski-Harabasz": ch,

        "Log-verossimilhança ajuste":
            tuning_log_likelihood,

        "Tempo (s)": training_time,

        "Convergência": gmm_model.converged_,

        "Iterações": gmm_model.n_iter_,

        "Clusters encontrados": n_labels,

        "Maior cluster (%)":
            percentages.max(),

        "Menor cluster (%)":
            percentages.min()
    })

    print(

        f"K={k} | "

        f"BIC={bic:.2f} | "

        f"Silhouette={silhouette:.4f} | "

        f"Convergência={gmm_model.converged_} | "

        f"Tempo={training_time:.2f}s"
    )

print("\nAvaliação do GMM concluída!")

In [ ]:
df_gmm_results = pd.DataFrame(
    gmm_results
)

display(
    df_gmm_results.round(4)
)

#### 9.3. Critérios AIC e BIC

Os critérios Akaike Information Criterion (AIC)
e Bayesian Information Criterion (BIC) são utilizados
para avaliar o ajuste probabilístico e a complexidade
dos modelos GMM.

In [ ]:
# Considerar somente modelos convergentes
df_gmm_converged = df_gmm_results[

    df_gmm_results["Convergência"]

    & np.isfinite(df_gmm_results["BIC"])

    & np.isfinite(df_gmm_results["AIC"])
].copy()

if df_gmm_converged.empty:

    raise ValueError(
        "Nenhum modelo GMM convergiu."
    )

# Menor AIC
best_k_aic = int(

    df_gmm_converged.loc[

        df_gmm_converged["AIC"].idxmin(),

        "K"
    ]
)

# Menor BIC
best_k_bic = int(

    df_gmm_converged.loc[

        df_gmm_converged["BIC"].idxmin(),

        "K"
    ]
)

print("Melhor número de componentes pelo AIC:")
print(best_k_aic)

print("\nMelhor número de componentes pelo BIC:")
print(best_k_bic)

In [ ]:
fig, axes = plt.subplots(

    1, 2,

    figsize=(14, 5)
)

# AIC
axes[0].plot(

    GMM_range,

    aic_scores,

    marker="o",

    color="steelblue"
)

axes[0].axvline(

    best_k_aic,

    color="red",

    linestyle="--",

    label=f"Melhor K = {best_k_aic}"
)

axes[0].set_title("Critério AIC")

axes[0].set_ylabel("AIC")

# BIC
axes[1].plot(

    GMM_range,

    bic_scores,

    marker="o",

    color="darkorange"
)

axes[1].axvline(

    best_k_bic,

    color="red",

    linestyle="--",

    label=f"Melhor K = {best_k_bic}"
)

axes[1].set_title("Critério BIC")

axes[1].set_ylabel("BIC")

for ax in axes:

    ax.set_xlabel(
        "Número de componentes"
    )

    ax.set_xticks(GMM_range)

    ax.grid(alpha=0.3)

    ax.legend()

plt.suptitle(
    "Seleção do Número de Componentes — GMM"
)

plt.tight_layout()
plt.show()

#### 9.4. Métricas complementares de agrupamento

In [ ]:
fig, axes = plt.subplots(

    2, 2,

    figsize=(14, 9)
)

# Métricas avaliadas
metrics_gmm = [

    (
        "Silhouette",
        "Coeficiente de Silhouette",
        "steelblue"
    ),

    (
        "Davies-Bouldin",
        "Índice Davies-Bouldin",
        "darkorange"
    ),

    (
        "Calinski-Harabasz",
        "Índice Calinski-Harabasz",
        "seagreen"
    ),

    (
        "Log-verossimilhança ajuste",
        "Log-verossimilhança média no ajuste",
        "slateblue"
    )
]

for ax, (column, title, color) in zip(

    axes.ravel(),

    metrics_gmm
):

    ax.plot(

        df_gmm_results["K"],

        df_gmm_results[column],

        marker="o",

        color=color
    )

    ax.axvline(

        best_k_bic,

        color="red",

        linestyle="--",

        alpha=0.5
    )

    ax.set_title(title)

    ax.set_xlabel(
        "Número de componentes"
    )

    ax.set_xticks(GMM_range)

    ax.grid(alpha=0.3)

plt.suptitle(
    "Métricas Complementares — GMM",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
gmm_distribution_results = []

for k in GMM_range:

    labels = gmm_evaluation_labels[k]

    counts = np.bincount(

        labels,

        minlength=k
    )

    for cluster_id in range(k):

        gmm_distribution_results.append({

            "K": k,

            "Componente": cluster_id,

            "Quantidade":
                counts[cluster_id],

            "Percentual (%)":
                100 * counts[cluster_id]
                / len(labels)
        })

df_gmm_all_distributions = pd.DataFrame(
    gmm_distribution_results
)

display(
    df_gmm_all_distributions.round(4)
)

In [ ]:
fig, axes = plt.subplots(

    2, 4,

    figsize=(18, 9),

    sharey=True
)

for ax, k in zip(

    axes.ravel(),

    GMM_range
):

    df_k = df_gmm_all_distributions[

        df_gmm_all_distributions["K"] == k
    ]

    colors_k = plt.get_cmap(

        "Spectral",

        k

    )(np.arange(k))

    bars = ax.bar(

        df_k["Componente"],

        df_k["Percentual (%)"],

        color=colors_k
    )

    ax.bar_label(

        bars,

        fmt="%.1f",

        fontsize=8
    )

    ax.set_title(f"K = {k}")

    ax.set_xlabel("Componente")

    ax.set_ylabel("Percentual (%)")

    ax.set_xticks(np.arange(k))

    ax.set_ylim(0, 110)

    ax.grid(

        axis="y",

        alpha=0.3
    )

plt.suptitle(
    "Distribuição dos Agrupamentos — GMM",
    fontsize=15
)

plt.tight_layout()
plt.show()

#### 9.5. Modelo definitivo

In [ ]:
# Número de componentes selecionado
GMM_IDEAL = best_k_bic

# Recuperar o modelo previamente treinado
final_gmm = gmm_models[GMM_IDEAL]

# Verificar convergência
assert final_gmm.converged_

# Rótulos do treinamento
labels_train_gmm = final_gmm.predict(
    X_train_final
)

# Rótulos do ajuste
labels_tuning_gmm = final_gmm.predict(
    X_tuning_final
)

# Tempo de treinamento
gmm_training_time = float(

    df_gmm_results.loc[

        df_gmm_results["K"] == GMM_IDEAL,

        "Tempo (s)"

    ].iloc[0]
)

print("Modelo GMM definitivo selecionado!")

print("\nNúmero de componentes:")
print(GMM_IDEAL)

print("\nDimensões das médias:")
print(final_gmm.means_.shape)

print("\nDimensões das covariâncias:")
print(final_gmm.covariances_.shape)

print("\nConvergência:")
print(final_gmm.converged_)

print("\nNúmero de iterações:")
print(final_gmm.n_iter_)

print(
    f"\nTempo de treinamento: "
    f"{gmm_training_time:.2f} segundos"
)

### 9.7. Caracterização estatística das componentes

As componentes gaussianas são caracterizadas por seus
vetores de médias, matrizes de covariância e pesos
probabilísticos.

Para facilitar a interpretação dos atributos sísmicos,
é realizada a transformação inversa da padronização
dos vetores de médias.

In [ ]:
# Médias no espaço padronizado
gmm_means_scaled = final_gmm.means_

# Recuperar escala anterior à padronização
gmm_means_original = (

    scaler_original.inverse_transform(

        gmm_means_scaled
    )
)

# Tabela das médias padronizadas
df_gmm_means_scaled = pd.DataFrame(

    gmm_means_scaled,

    columns=selected_features
)

df_gmm_means_scaled.index.name = "Componente"

# Tabela das médias recuperadas
df_gmm_means = pd.DataFrame(

    gmm_means_original,

    columns=selected_features
)

df_gmm_means.index.name = "Componente"

# Pesos probabilísticos das componentes
df_gmm_means["Peso probabilístico (%)"] = (

    final_gmm.weights_ * 100
)

print("Médias padronizadas:")

display(
    df_gmm_means_scaled.round(4)
)

print("\nMédias na escala dos atributos:")

display(
    df_gmm_means.round(4)
)

In [ ]:
# Desvios padrões no espaço padronizado
gmm_std_scaled = np.sqrt(

    np.diagonal(

        final_gmm.covariances_,

        axis1=1,

        axis2=2
    )
)

# Recuperar escala anterior à padronização
gmm_std_original = (

    gmm_std_scaled

    * scaler_original.scale_
)

df_gmm_std = pd.DataFrame(

    gmm_std_original,

    columns=[

        "Desvio amplitude",

        "Desvio envelope",

        "Desvio fase"
    ]
)

df_gmm_std.index.name = "Componente"

display(
    df_gmm_std.round(4)
)

In [ ]:
# Mapa de cores exclusivo do GMM
gmm_colors = plt.get_cmap(

    "Spectral",

    GMM_IDEAL
)

gmm_color_values = gmm_colors(

    np.arange(GMM_IDEAL)
)

# Gráfico de barras
df_gmm_means_scaled.T.plot(

    kind="bar",

    figsize=(12, 6),

    color=gmm_color_values
)

plt.title(

    f"Caracterização das Componentes "

    f"Gaussianas (K={GMM_IDEAL})"
)

plt.xlabel("Atributo sísmico")

plt.ylabel("Média padronizada")

plt.axhline(

    0,

    color="black",

    linewidth=0.8
)

plt.xticks(rotation=0)

plt.legend(title="Componente")

plt.grid(

    axis="y",

    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
# Organização dos gráficos
n_cols = min(3, GMM_IDEAL)

n_rows = (

    GMM_IDEAL + n_cols - 1

) // n_cols

fig, axes = plt.subplots(

    n_rows,

    n_cols,

    figsize=(5 * n_cols, 4 * n_rows),

    squeeze=False
)

axes = axes.ravel()

# Limite comum para todas as matrizes
cov_limit = np.max(

    np.abs(
        final_gmm.covariances_
    )
)

cov_limit = max(
    cov_limit,
    1e-12
)

for k in range(GMM_IDEAL):

    covariance = final_gmm.covariances_[k]

    ax = axes[k]

    im = ax.imshow(

        covariance,

        cmap="coolwarm",

        vmin=-cov_limit,

        vmax=cov_limit
    )

    ax.set_xticks(np.arange(3))

    ax.set_yticks(np.arange(3))

    ax.set_xticklabels(

        selected_features,

        rotation=45,

        ha="right"
    )

    ax.set_yticklabels(
        selected_features
    )

    for i in range(3):

        for j in range(3):

            ax.text(

                j,

                i,

                f"{covariance[i, j]:.2f}",

                ha="center",

                va="center",

                color="white"
                if abs(covariance[i, j]) > 0.6 * cov_limit
                else "black"
            )

    ax.set_title(
        f"Componente {k}"
    )

    fig.colorbar(

        im,

        ax=ax,

        fraction=0.046
    )

# Remover gráficos excedentes
for ax in axes[GMM_IDEAL:]:

    ax.axis("off")

plt.suptitle(

    "Matrizes de Covariância das Componentes",

    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
df_gmm_attributes = pd.DataFrame(

    X_train_original,

    columns=selected_features
)

df_gmm_attributes["Componente"] = (

    labels_train_gmm
)

df_gmm_cluster_statistics = (

    df_gmm_attributes

    .groupby("Componente")[selected_features]

    .agg(["mean", "std", "min", "max"])
)

display(
    df_gmm_cluster_statistics.round(4)
)

#### 9.7. Distribuição percentual dos agrupamentos

In [ ]:
# Contagem no treinamento
gmm_counts_train = np.bincount(

    labels_train_gmm,

    minlength=GMM_IDEAL
)

# Contagem no ajuste
gmm_counts_tuning = np.bincount(

    labels_tuning_gmm,

    minlength=GMM_IDEAL
)

# Tabela de distribuição
df_gmm_distribution = pd.DataFrame({

    "Componente":
        np.arange(GMM_IDEAL),

    "Treinamento":
        gmm_counts_train,

    "Treinamento (%)":
        100 * gmm_counts_train
        / len(labels_train_gmm),

    "Ajuste":
        gmm_counts_tuning,

    "Ajuste (%)":
        100 * gmm_counts_tuning
        / len(labels_tuning_gmm),

    "Peso probabilístico (%)":
        final_gmm.weights_ * 100
})

display(
    df_gmm_distribution.round(4)
)

In [ ]:
fig, axes = plt.subplots(

    1, 3,

    figsize=(17, 5),

    sharey=True
)

distribution_columns = [

    (
        "Treinamento (%)",
        "Distribuição — Treinamento"
    ),

    (
        "Ajuste (%)",
        "Distribuição — Ajuste"
    ),

    (
        "Peso probabilístico (%)",
        "Pesos das Gaussianas"
    )
]

for ax, (column, title) in zip(

    axes,

    distribution_columns
):

    bars = ax.bar(

        np.arange(GMM_IDEAL),

        df_gmm_distribution[column],

        color=gmm_color_values
    )

    ax.bar_label(

        bars,

        fmt="%.2f%%",

        fontsize=8
    )

    ax.set_title(title)

    ax.set_xlabel("Componente")

    ax.set_ylabel("Percentual (%)")

    ax.set_xticks(
        np.arange(GMM_IDEAL)
    )

    ax.set_ylim(0, 110)

    ax.grid(

        axis="y",

        alpha=0.3
    )

plt.tight_layout()
plt.show()

#### 9.8. Visualização dos agrupamentos

Os agrupamentos identificados pelo GMM são visualizados
diretamente no espaço da amplitude, envelope e fase
instantânea.

In [ ]:
from matplotlib.colors import BoundaryNorm

# Mesmas amostras utilizadas no K-means
gmm_visual_indices = visual_indices.copy()

# Atributos antes da padronização
X_gmm_visual = X_train_original[
    gmm_visual_indices
]

# Rótulos correspondentes
labels_gmm_visual = labels_train_gmm[
    gmm_visual_indices
]

# Normalização das cores
gmm_norm = BoundaryNorm(

    np.arange(

        -0.5,

        GMM_IDEAL + 0.5,

        1
    ),

    GMM_IDEAL
)

print(

    "Amostras utilizadas na visualização:",

    len(gmm_visual_indices)
)

In [ ]:
fig = plt.figure(

    figsize=(12, 8)
)

ax = fig.add_subplot(

    111,

    projection="3d"
)

scatter = ax.scatter(

    X_gmm_visual[:, 0],

    X_gmm_visual[:, 1],

    X_gmm_visual[:, 2],

    c=labels_gmm_visual,

    cmap=gmm_colors,

    norm=gmm_norm,

    s=6,

    alpha=0.5
)

# Médias das gaussianas
ax.scatter(

    gmm_means_original[:, 0],

    gmm_means_original[:, 1],

    gmm_means_original[:, 2],

    marker="X",

    c="black",

    s=150,

    label="Médias das gaussianas"
)

ax.set_xlabel(
    "Amplitude normalizada"
)

ax.set_ylabel("Envelope")

ax.set_zlabel(
    "Fase instantânea (rad)"
)

ax.set_title(

    f"Agrupamento Tridimensional "

    f"GMM (K={GMM_IDEAL})"
)

ax.legend()

fig.colorbar(

    scatter,

    ax=ax,

    ticks=np.arange(GMM_IDEAL),

    label="Componente",

    shrink=0.7
)

plt.tight_layout()
plt.show()

In [ ]:
gmm_feature_pairs = [

    ("Amplitude", "Envelope"),

    ("Amplitude", "Fase"),

    ("Envelope", "Fase")
]

fig, axes = plt.subplots(

    1, 3,

    figsize=(18, 5)
)

for ax, (feature_x, feature_y) in zip(

    axes,

    gmm_feature_pairs
):

    idx_x = selected_features.index(
        feature_x
    )

    idx_y = selected_features.index(
        feature_y
    )

    scatter = ax.scatter(

        X_gmm_visual[:, idx_x],

        X_gmm_visual[:, idx_y],

        c=labels_gmm_visual,

        cmap=gmm_colors,

        norm=gmm_norm,

        s=5,

        alpha=0.5
    )

    # Médias das gaussianas
    ax.scatter(

        gmm_means_original[:, idx_x],

        gmm_means_original[:, idx_y],

        marker="X",

        c="black",

        s=100
    )

    ax.set_xlabel(feature_x)

    ax.set_ylabel(feature_y)

    ax.set_title(

        f"{feature_x} × {feature_y}"
    )

    ax.grid(alpha=0.2)

fig.colorbar(

    scatter,

    ax=axes,

    ticks=np.arange(GMM_IDEAL),

    label="Componente",

    shrink=0.8
)

fig.suptitle(

    f"Relações entre os Atributos "

    f"Sísmicos — GMM (K={GMM_IDEAL})",

    fontsize=14
)

fig.subplots_adjust(

    top=0.85,

    wspace=0.35,

    right=0.90
)

plt.show()

#### 9.9. Análise das probabilidades e incertezas

O GMM permite estimar probabilidades de pertencimento
das amostras às diferentes componentes gaussianas.

A maior probabilidade posterior é utilizada como
indicador do grau de pertencimento da amostra à
componente selecionada.

In [ ]:
# Probabilidades do conjunto de ajuste
gmm_probabilities = final_gmm.predict_proba(

    X_tuning_final
)

# Maior probabilidade de cada amostra
gmm_confidence = np.max(

    gmm_probabilities,

    axis=1
)

# Componente com maior probabilidade
gmm_predicted_components = np.argmax(

    gmm_probabilities,

    axis=1
)

# Verificar consistência das probabilidades
assert np.allclose(

    gmm_probabilities.sum(axis=1),

    1
)

assert np.array_equal(

    gmm_predicted_components,

    labels_tuning_gmm
)

print(

    "Formato da matriz de probabilidades:",

    gmm_probabilities.shape
)

# Visualizar primeiras probabilidades
df_gmm_probabilities = pd.DataFrame(

    gmm_probabilities,

    columns=[

        f"Componente {k}"

        for k in range(GMM_IDEAL)
    ]
)

display(

    df_gmm_probabilities.head(10).round(4)
)

In [ ]:
# Evitar logaritmo de zero
safe_probabilities = np.clip(

    gmm_probabilities,

    1e-12,

    1
)

# Entropia normalizada
gmm_entropy = (

    -np.sum(

        gmm_probabilities

        * np.log(safe_probabilities),

        axis=1

    ) / np.log(GMM_IDEAL)
)

# Corrigir pequenos desvios numéricos
gmm_entropy = np.clip(

    gmm_entropy,

    0,

    1
)

df_gmm_uncertainty = pd.DataFrame({

    "Probabilidade máxima":
        gmm_confidence,

    "Entropia normalizada":
        gmm_entropy
})

display(

    df_gmm_uncertainty.describe().T.round(4)
)

print(

    "\nProbabilidade máxima média:",

    round(gmm_confidence.mean(), 4)
)

print(

    "Entropia média:",

    round(gmm_entropy.mean(), 4)
)

print(

    "Amostras com probabilidade máxima inferior a 70%:",

    round(

        100 * np.mean(gmm_confidence < 0.7),

        2
    ),

    "%"
)

In [ ]:
fig, axes = plt.subplots(

    1, 2,

    figsize=(14, 5)
)

# Probabilidade máxima
axes[0].hist(

    gmm_confidence,

    bins=40,

    color="steelblue",

    edgecolor="white"
)

axes[0].axvline(

    gmm_confidence.mean(),

    color="red",

    linestyle="--",

    label="Média"
)

axes[0].set_title(

    "Distribuição da Probabilidade Máxima"
)

axes[0].set_xlabel(

    "Probabilidade de pertencimento"
)

# Entropia
axes[1].hist(

    gmm_entropy,

    bins=40,

    color="darkorange",

    edgecolor="white"
)

axes[1].axvline(

    gmm_entropy.mean(),

    color="red",

    linestyle="--",

    label="Média"
)

axes[1].set_title(

    "Distribuição da Entropia Normalizada"
)

axes[1].set_xlabel("Entropia")

for ax in axes:

    ax.set_ylabel("Quantidade de amostras")

    ax.set_xlim(0, 1)

    ax.legend()

    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

#### 9.11. Segmentação sísmica da Bacia do Jequitinhonha

In [ ]:
print(

    f"Segmentação GMM da Bacia do Jequitinhonha "

    f"com K={GMM_IDEAL}"
)

start_time = time.perf_counter()

# Reutilização da função da etapa 8
segmentation_volume_gmm = segment_seismic_data(

    seismic_data,

    dt_jequitinhonha,

    final_gmm,

    final_preprocessor,

    selected_indices
)

# Tempo de segmentação
gmm_segmentation_time = (

    time.perf_counter() - start_time
)

# Rótulos em uma dimensão
cluster_labels_flat_gmm = (

    segmentation_volume_gmm.reshape(-1)
)

# Verificação das dimensões
assert (

    segmentation_volume_gmm.shape

    == seismic_data.shape
)

# Verificar rótulos válidos
assert np.all(

    (cluster_labels_flat_gmm >= -1)

    & (cluster_labels_flat_gmm < GMM_IDEAL)
)

print("\nSegmentação GMM concluída!")

print(

    "Dimensões originais:",

    seismic_data.shape
)

print(

    "Dimensões da segmentação:",

    segmentation_volume_gmm.shape
)

print(

    "Rótulos encontrados:",

    np.unique(cluster_labels_flat_gmm)
)

print(

    f"Tempo de segmentação: "

    f"{gmm_segmentation_time:.2f} segundos"
)

In [ ]:
# Amostras classificadas
valid_labels_gmm = cluster_labels_flat_gmm[

    cluster_labels_flat_gmm >= 0
]

# Contagem por componente
gmm_cluster_counts = np.bincount(

    valid_labels_gmm,

    minlength=GMM_IDEAL
)

# Quantidade de amostras
gmm_total_voxels = len(

    cluster_labels_flat_gmm
)

gmm_valid_voxels = len(

    valid_labels_gmm
)

gmm_invalid_voxels = (

    gmm_total_voxels

    - gmm_valid_voxels
)

if gmm_valid_voxels == 0:

    raise ValueError(

        "Nenhuma amostra foi classificada pelo GMM."
    )

# Acrescentar distribuição à tabela das médias
df_gmm_means["Quantidade de amostras"] = (

    gmm_cluster_counts
)

df_gmm_means["% Volume Total"] = (

    100 * gmm_cluster_counts

    / gmm_total_voxels
)

df_gmm_means["% Volume Classificado"] = (

    100 * gmm_cluster_counts

    / gmm_valid_voxels
)

display(

    df_gmm_means.round(4)
)

print(

    "Total de amostras:",

    gmm_total_voxels
)

print(

    "Amostras classificadas:",

    gmm_valid_voxels
)

print(

    "Amostras não classificadas:",

    gmm_invalid_voxels
)

print(

    "Percentual classificado:",

    round(

        100 * gmm_valid_voxels

        / gmm_total_voxels,

        4
    )
)

In [ ]:
# Segmentação em duas dimensões
segmentation_volume_gmm_2d = (

    segmentation_volume_gmm.reshape(

        -1,

        segmentation_volume_gmm.shape[2]
    )
)

# Mapa de cores
cmap_gmm_spectral = plt.get_cmap(

    "Spectral",

    GMM_IDEAL

).copy()

# Regiões não classificadas
cmap_gmm_spectral.set_bad(

    color="lightgray"
)

# Máscara das regiões inválidas
segmentation_gmm_masked = np.ma.masked_equal(

    segmentation_volume_gmm_2d,

    -1
)

print(

    "Dimensões da segmentação:",

    segmentation_volume_gmm_2d.shape
)

In [ ]:
v_limit_gmm = max(

    np.percentile(

        np.abs(seismic_data_2d),

        99.5
    ),

    1e-12
)

fig, axes = plt.subplots(

    1, 2,

    figsize=(18, 7),

    sharey=True
)

# Amplitude original
im1 = axes[0].imshow(

    seismic_data_2d.T,

    cmap="seismic",

    vmin=-v_limit_gmm,

    vmax=v_limit_gmm,

    aspect="auto",

    interpolation="nearest"
)

axes[0].set_title(

    "Amplitude Sísmica Original"
)

# Segmentação GMM
im2 = axes[1].imshow(

    segmentation_gmm_masked.T,

    cmap=cmap_gmm_spectral,

    norm=gmm_norm,

    aspect="auto",

    interpolation="nearest"
)

axes[1].set_title(

    f"Segmentação GMM (K={GMM_IDEAL})"
)

for ax in axes:

    ax.set_xlabel("Índice do traço")

axes[0].set_ylabel(

    "Amostra temporal"
)

fig.colorbar(

    im1,

    ax=axes[0],

    label="Amplitude",

    fraction=0.046
)

fig.colorbar(

    im2,

    ax=axes[1],

    ticks=np.arange(GMM_IDEAL),

    label="Componente",

    fraction=0.046
)

plt.suptitle(

    "Segmentação GMM — Bacia do Jequitinhonha",

    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Número de traços exibidos
n_display_gmm = min(
    MAX_DISPLAY_TRACES,
    seismic_data_2d.shape[0]
)

# Selecionar o bloco central
gmm_display_start = (
    seismic_data_2d.shape[0] - n_display_gmm
) // 2

gmm_display_stop = (
    gmm_display_start + n_display_gmm
)

# Selecionar amplitude original
gmm_amplitude_detail = seismic_data_2d[
    gmm_display_start:gmm_display_stop
]

# Selecionar segmentação correspondente
gmm_segmentation_detail = np.ma.masked_equal(
    segmentation_volume_gmm_2d[
        gmm_display_start:gmm_display_stop
    ],
    -1
)

# Limite de contraste
gmm_detail_limit = max(
    np.percentile(
        np.abs(gmm_amplitude_detail),
        99.5
    ),
    1e-12
)

# Criação dos gráficos
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 7),
    sharey=True
)

# Amplitude original
axes[0].imshow(
    gmm_amplitude_detail.T,
    cmap="seismic",
    vmin=-gmm_detail_limit,
    vmax=gmm_detail_limit,
    aspect="auto",
    interpolation="nearest"
)

axes[0].set_title(
    "Amplitude Original — Detalhe"
)

# Segmentação GMM
im = axes[1].imshow(
    gmm_segmentation_detail.T,
    cmap=cmap_gmm_spectral,
    norm=gmm_norm,
    aspect="auto",
    interpolation="nearest"
)

axes[1].set_title(
    f"Segmentação GMM — Detalhe (K={GMM_IDEAL})"
)

# Configuração dos eixos
for ax in axes:
    ax.set_xlabel("Índice do traço no bloco")

axes[0].set_ylabel("Amostra temporal")

# Barra de cores
fig.colorbar(
    im,
    ax=axes[1],
    ticks=np.arange(GMM_IDEAL),
    label="Componente"
)

plt.suptitle(
    f"Detalhamento dos Traços "
    f"{gmm_display_start} a {gmm_display_stop - 1}",
    fontsize=14
)

plt.tight_layout()
plt.show()

#### 9.11. Visualização individual dos agrupamentos

In [ ]:
from matplotlib.colors import ListedColormap

def plot_individual_gmm_clusters():

    # Mesma subamostragem do K-means
    sample_step_traces = 4

    sample_step_samples = 2

    # Seleção das amostras
    seg_sampled_gmm = segmentation_volume_gmm_2d[

        ::sample_step_traces,

        ::sample_step_samples
    ]

    # Organização dos gráficos
    n_cols = min(3, GMM_IDEAL)

    n_rows = (

        GMM_IDEAL + n_cols - 1

    ) // n_cols

    fig, axes = plt.subplots(

        n_rows,

        n_cols,

        figsize=(5.5 * n_cols, 4 * n_rows),

        squeeze=False
    )

    axes = axes.ravel()

    for cluster_id in range(GMM_IDEAL):

        # Máscara binária
        cluster_mask = (

            seg_sampled_gmm == cluster_id
        )

        # Cor da componente
        individual_cmap = ListedColormap([

            "white",

            gmm_color_values[cluster_id]
        ])

        ax = axes[cluster_id]

        ax.imshow(

            cluster_mask.T,

            cmap=individual_cmap,

            vmin=0,

            vmax=1,

            aspect="auto",

            interpolation="nearest",

            extent=(

                -0.5 * sample_step_traces,

                (seg_sampled_gmm.shape[0] - 0.5)
                * sample_step_traces,

                (seg_sampled_gmm.shape[1] - 0.5)
                * sample_step_samples,

                -0.5 * sample_step_samples
            )
        )

        # Percentual classificado
        percentage = df_gmm_means.loc[

            cluster_id,

            "% Volume Classificado"
        ]

        ax.set_title(

            f"Componente {cluster_id} — "

            f"{percentage:.2f}%"
        )

        ax.set_xlabel(

            "Índice do traço"
        )

        ax.set_ylabel(

            "Amostra temporal"
        )

    # Remover eixos excedentes
    for ax in axes[GMM_IDEAL:]:

        ax.axis("off")

    plt.suptitle(

        f"Visualização Individual das Componentes "

        f"GMM (K={GMM_IDEAL})",

        fontsize=15
    )

    plt.tight_layout()
    plt.show()

In [ ]:
plot_individual_gmm_clusters()

### 10. Validação Interna dos Modelos

Após o treinamento dos algoritmos K-means e Gaussian
Mixture Model (GMM), é realizada uma avaliação sobre
o conjunto de validação interna da Bacia do Jequitinhonha.

O conjunto de validação contém amostras provenientes
de traços que não participaram do treinamento nem da
seleção dos hiperparâmetros.

#### 10.1. Preparação dos dados para validação

São utilizadas as amostras do conjunto de validação
interna, previamente transformadas pelo padronizador
ajustado sobre os dados de treinamento.

In [ ]:
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Verificar atributos utilizados
assert selected_features == [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Verificar os modelos treinados
assert final_kmeans.n_features_in_ == 3

assert final_gmm.n_features_in_ == 3

assert final_gmm.converged_

# Verificar o pré-processador
assert scaler.n_features_in_ == 7

assert scaler_original.n_features_in_ == 3

assert final_preprocessor is scaler_original

# Verificar conjunto de validação
assert X_val_final.shape[1] == 3

assert np.isfinite(X_val_final).all()

print("Dados verificados com sucesso!")

print("\nAtributos utilizados:")
print(selected_features)

print("\nConjunto de validação:")
print(X_val_final.shape)

print("\nNúmero de clusters do K-means:")
print(K_IDEAL)

print("\nNúmero de componentes do GMM:")
print(GMM_IDEAL)

In [ ]:
# Predição utilizando o K-means
labels_val_kmeans = final_kmeans.predict(
    X_val_final
)

# Predição utilizando o GMM
labels_val_gmm = final_gmm.predict(
    X_val_final
)

# Verificar quantidade de rótulos
assert len(labels_val_kmeans) == len(X_val_final)

assert len(labels_val_gmm) == len(X_val_final)

print("Predições concluídas!")

print("\nK-means:")

print("Amostras:", len(labels_val_kmeans))

print(
    "Clusters encontrados:",
    np.unique(labels_val_kmeans)
)

print("\nGMM:")

print("Amostras:", len(labels_val_gmm))

print(
    "Componentes encontradas:",
    np.unique(labels_val_gmm)
)

In [ ]:
# Gerador específico da validação
validation_rng = np.random.default_rng(
    RANDOM_STATE + 10
)

# Selecionar as mesmas amostras para os dois modelos
validation_evaluation_indices = validation_rng.choice(

    len(X_val_final),

    size=min(
        SILHOUETTE_SAMPLES,
        len(X_val_final)
    ),

    replace=False
)

# Atributos selecionados
X_validation_evaluation = X_val_final[
    validation_evaluation_indices
]

# Rótulos do K-means
labels_val_kmeans_eval = labels_val_kmeans[
    validation_evaluation_indices
]

# Rótulos do GMM
labels_val_gmm_eval = labels_val_gmm[
    validation_evaluation_indices
]

print(
    "Total de amostras de validação:",
    len(X_val_final)
)

print(
    "Amostras utilizadas nas métricas:",
    len(X_validation_evaluation)
)

#### 10.2. Avaliação quantitativa dos modelos

In [ ]:
def evaluate_clustering(X, labels):

    # Número de amostras
    n_samples = len(X)

    # Número de agrupamentos encontrados
    n_clusters = len(
        np.unique(labels)
    )

    # Verificar consistência dos dados
    if len(labels) != n_samples:
        raise ValueError(
            "Quantidade de amostras e rótulos incompatível."
        )

    # Verificar se as métricas podem ser calculadas
    if not 2 <= n_clusters < n_samples:

        return {
            "Silhouette": np.nan,
            "Davies-Bouldin": np.nan,
            "Calinski-Harabasz": np.nan,
            "Clusters encontrados": n_clusters
        }

    return {

        "Silhouette": silhouette_score(
            X,
            labels
        ),

        "Davies-Bouldin": davies_bouldin_score(
            X,
            labels
        ),

        "Calinski-Harabasz": calinski_harabasz_score(
            X,
            labels
        ),

        "Clusters encontrados": n_clusters
    }

In [ ]:
# Avaliação do K-means
metrics_val_kmeans = evaluate_clustering(

    X_validation_evaluation,

    labels_val_kmeans_eval
)

# Avaliação do GMM
metrics_val_gmm = evaluate_clustering(

    X_validation_evaluation,

    labels_val_gmm_eval
)

# Organização dos resultados
df_internal_validation = pd.DataFrame([

    {

        "Modelo": "K-means",

        "K": K_IDEAL,

        **metrics_val_kmeans
    },

    {

        "Modelo": "GMM",

        "K": GMM_IDEAL,

        **metrics_val_gmm
    }
])

display(
    df_internal_validation.round(4)
)

In [ ]:
metric_directions = {

    "Silhouette": "max",

    "Davies-Bouldin": "min",

    "Calinski-Harabasz": "max"
}

for metric, direction in metric_directions.items():

    valid_results = df_internal_validation.dropna(
        subset=[metric]
    )

    if valid_results.empty:

        print(f"{metric}: resultado indisponível")
        continue

    if direction == "max":

        best_index = valid_results[metric].idxmax()

    else:

        best_index = valid_results[metric].idxmin()

    best_model = valid_results.loc[
        best_index,
        "Modelo"
    ]

    best_value = valid_results.loc[
        best_index,
        metric
    ]

    print(

        f"{metric}: "

        f"{best_model} "

        f"({best_value:.4f})"
    )

#### 10.3. Comparação gráfica das métricas

Os resultados da validação interna são apresentados
graficamente para facilitar a comparação entre os
algoritmos K-means e GMM.

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

validation_metrics = [

    ("Silhouette", "Silhouette — Maior é melhor"),

    ("Davies-Bouldin", "Davies-Bouldin — Menor é melhor"),

    ("Calinski-Harabasz", "Calinski-Harabasz — Maior é melhor")
]

model_colors = [
    "steelblue",
    "darkorange"
]

for ax, (metric, title) in zip(

    axes,

    validation_metrics
):

    bars = ax.bar(

        df_internal_validation["Modelo"],

        df_internal_validation[metric],

        color=model_colors,

        width=0.55
    )

    ax.bar_label(

        bars,

        fmt="%.4g",

        padding=4
    )

    ax.set_title(title)

    ax.set_ylabel(metric)

    ax.margins(y=0.15)

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.suptitle(

    "Comparação dos Modelos — Validação Interna",

    fontsize=15
)

plt.tight_layout()
plt.show()

#### 10.5. Comparação com o mesmo número de agrupamentos

É realizada uma comparação adicional entre os algoritmos
considerando o mesmo número de agrupamentos.

O número de clusters selecionado pelo K-means é utilizado
como referência para essa comparação.

In [ ]:
# Número de clusters para comparação controlada
K_COMMON = K_IDEAL

# Recuperar modelos previamente treinados
kmeans_common = kmeans_models[K_COMMON]
gmm_common = gmm_models[K_COMMON]

# Verificar convergência
assert gmm_common.converged_

# Predições na validação
labels_val_kmeans_common = kmeans_common.predict(
    X_val_final
)

labels_val_gmm_common = gmm_common.predict(
    X_val_final
)

print("Comparação controlada preparada!")
print("Número de agrupamentos:", K_COMMON)

In [ ]:
# Avaliação do K-means
metrics_kmeans_common = evaluate_clustering(
    X_validation_evaluation,
    labels_val_kmeans_common[
        validation_evaluation_indices
    ]
)

# Avaliação do GMM
metrics_gmm_common = evaluate_clustering(
    X_validation_evaluation,
    labels_val_gmm_common[
        validation_evaluation_indices
    ]
)

# Construção da tabela comparativa
df_common_k_validation = pd.DataFrame([
    {
        "Modelo": "K-means",
        "K": K_COMMON,
        **metrics_kmeans_common
    },
    {
        "Modelo": "GMM",
        "K": K_COMMON,
        **metrics_gmm_common
    }
])

display(df_common_k_validation.round(4))

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(16, 5)
)

for ax, (metric, title) in zip(

    axes,

    validation_metrics
):

    bars = ax.bar(

        df_common_k_validation["Modelo"],

        df_common_k_validation[metric],

        color=model_colors,

        width=0.55
    )

    ax.bar_label(

        bars,

        fmt="%.4g",

        padding=4
    )

    ax.set_title(title)

    ax.set_ylabel(metric)

    ax.margins(y=0.15)

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.suptitle(

    f"Comparação Controlada dos Modelos — K={K_COMMON}",

    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
df_comparison_final = pd.concat([

    df_internal_validation.assign(
        Experimento="Configurações selecionadas"
    ),

    df_common_k_validation.assign(
        Experimento="Mesmo K"
    )

], ignore_index=True)

df_comparison_final = df_comparison_final[[

    "Experimento",

    "Modelo",

    "K",

    "Silhouette",

    "Davies-Bouldin",

    "Calinski-Harabasz",

    "Clusters encontrados"
]]

display(
    df_comparison_final.round(4)
)

#### 10.5. Distribuição dos agrupamentos

A distribuição percentual das amostras é analisada
nos conjuntos de treinamento, ajuste e validação interna.

In [ ]:
def calculate_cluster_distribution(

    labels,

    n_clusters
):

    counts = np.bincount(

        labels,

        minlength=n_clusters
    )

    percentages = (

        100 * counts / len(labels)
    )

    return percentages

In [ ]:
df_validation_distribution_kmeans = pd.DataFrame({

    "Cluster":
        np.arange(K_IDEAL),

    "Treinamento (%)":
        calculate_cluster_distribution(
            labels_train_kmeans,
            K_IDEAL
        ),

    "Ajuste (%)":
        calculate_cluster_distribution(
            labels_tuning_kmeans,
            K_IDEAL
        ),

    "Validação (%)":
        calculate_cluster_distribution(
            labels_val_kmeans,
            K_IDEAL
        )
})

display(
    df_validation_distribution_kmeans.round(4)
)

In [ ]:
df_validation_distribution_gmm = pd.DataFrame({

    "Componente":
        np.arange(GMM_IDEAL),

    "Treinamento (%)":
        calculate_cluster_distribution(
            labels_train_gmm,
            GMM_IDEAL
        ),

    "Ajuste (%)":
        calculate_cluster_distribution(
            labels_tuning_gmm,
            GMM_IDEAL
        ),

    "Validação (%)":
        calculate_cluster_distribution(
            labels_val_gmm,
            GMM_IDEAL
        )
})

display(
    df_validation_distribution_gmm.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 6),
    sharey=True
)

# K-means
df_validation_distribution_kmeans.set_index(

    "Cluster"

).plot(

    kind="bar",

    ax=axes[0],

    color=[
        "steelblue",
        "darkorange",
        "seagreen"
    ]
)

axes[0].set_title(
    f"Distribuição K-means (K={K_IDEAL})"
)

axes[0].set_xlabel("Cluster")

# GMM
df_validation_distribution_gmm.set_index(

    "Componente"

).plot(

    kind="bar",

    ax=axes[1],

    color=[
        "steelblue",
        "darkorange",
        "seagreen"
    ]
)

axes[1].set_title(
    f"Distribuição GMM (K={GMM_IDEAL})"
)

axes[1].set_xlabel("Componente")

for ax in axes:

    ax.set_ylabel("Percentual de amostras (%)")

    ax.tick_params(
        axis="x",
        rotation=0
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.legend()

plt.suptitle(

    "Distribuição dos Agrupamentos nos Conjuntos de Dados",

    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
def total_variation_distance(p, q):

    p = np.asarray(p, dtype=float)

    q = np.asarray(q, dtype=float)

    # Converter percentuais em proporções
    p = p / p.sum()

    q = q / q.sum()

    return 0.5 * np.abs(p - q).sum()


# K-means
tv_kmeans = total_variation_distance(

    df_validation_distribution_kmeans[
        "Treinamento (%)"
    ],

    df_validation_distribution_kmeans[
        "Validação (%)"
    ]
)

# GMM
tv_gmm = total_variation_distance(

    df_validation_distribution_gmm[
        "Treinamento (%)"
    ],

    df_validation_distribution_gmm[
        "Validação (%)"
    ]
)

df_distribution_shift = pd.DataFrame({

    "Modelo": [
        "K-means",
        "GMM"
    ],

    "Distância de variação total": [
        tv_kmeans,
        tv_gmm
    ]
})

display(
    df_distribution_shift.round(4)
)

#### 10.7. Concordância entre os modelos

A concordância entre os agrupamentos produzidos pelos
algoritmos K-means e GMM é avaliada utilizando o Adjusted
Rand Index (ARI) e a Normalized Mutual Information (NMI).

In [ ]:
# Modelos principais
ari_validation = adjusted_rand_score(

    labels_val_kmeans,

    labels_val_gmm
)

nmi_validation = normalized_mutual_info_score(

    labels_val_kmeans,

    labels_val_gmm
)

# Modelos com mesmo K
ari_common = adjusted_rand_score(

    labels_val_kmeans_common,

    labels_val_gmm_common
)

nmi_common = normalized_mutual_info_score(

    labels_val_kmeans_common,

    labels_val_gmm_common
)

# Resultados
df_model_agreement = pd.DataFrame({

    "Experimento": [

        "Configurações selecionadas",

        "Mesmo K"
    ],

    "ARI": [

        ari_validation,

        ari_common
    ],

    "NMI": [

        nmi_validation,

        nmi_common
    ]
})

display(
    df_model_agreement.round(4)
)

In [ ]:
# Tabela de correspondência
df_cluster_correspondence = pd.crosstab(

    pd.Series(

        labels_val_kmeans,

        name="K-means"
    ),

    pd.Series(

        labels_val_gmm,

        name="GMM"
    )
)

# Organizar todos os rótulos
df_cluster_correspondence = (
    df_cluster_correspondence.reindex(

        index=np.arange(K_IDEAL),

        columns=np.arange(GMM_IDEAL),

        fill_value=0
    )
)

display(
    df_cluster_correspondence
)

In [ ]:
# Percentual relativo a cada cluster do K-means
row_totals = df_cluster_correspondence.sum(
    axis=1
)

df_correspondence_percentage = (

    df_cluster_correspondence.div(

        row_totals.replace(0, np.nan),

        axis=0

    ) * 100
)

display(
    df_correspondence_percentage.round(2)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 7)
)

im = ax.imshow(

    df_correspondence_percentage.to_numpy(),

    cmap="Blues",

    vmin=0,

    vmax=100,

    aspect="auto"
)

ax.set_xticks(
    np.arange(GMM_IDEAL)
)

ax.set_yticks(
    np.arange(K_IDEAL)
)

ax.set_xlabel("Componente GMM")

ax.set_ylabel("Cluster K-means")

for i in range(K_IDEAL):

    for j in range(GMM_IDEAL):

        value = df_correspondence_percentage.iloc[
            i, j
        ]

        if np.isfinite(value):

            ax.text(

                j,

                i,

                f"{value:.1f}%",

                ha="center",

                va="center",

                color="white"
                if value > 50
                else "black"
            )

fig.colorbar(

    im,

    ax=ax,

    label="Percentual (%)"
)

ax.set_title(

    "Correspondência entre os Agrupamentos — Validação Interna"
)

plt.tight_layout()
plt.show()

#### 10.7. Análise das probabilidades e incertezas

As probabilidades de pertencimento fornecidas pelo GMM
são analisadas no conjunto de validação interna.

In [ ]:
# Probabilidades de pertencimento
gmm_probabilities_val = final_gmm.predict_proba(

    X_val_final
)

# Probabilidade máxima
gmm_confidence_val = np.max(

    gmm_probabilities_val,

    axis=1
)

# Entropia normalizada
gmm_entropy_val = (

    -np.sum(

        gmm_probabilities_val

        * np.log(

            np.clip(
                gmm_probabilities_val,
                1e-12,
                1
            )
        ),

        axis=1

    ) / np.log(GMM_IDEAL)
)

gmm_entropy_val = np.clip(

    gmm_entropy_val,

    0,

    1
)

# Log-verossimilhança média
gmm_log_likelihood_val = final_gmm.score(

    X_val_final
)

# Verificações
assert np.allclose(

    gmm_probabilities_val.sum(axis=1),

    1
)

assert np.array_equal(

    gmm_probabilities_val.argmax(axis=1),

    labels_val_gmm
)

print("Probabilidades calculadas!")

print(

    "\nProbabilidade máxima média:",

    round(gmm_confidence_val.mean(), 4)
)

print(

    "Entropia média:",

    round(gmm_entropy_val.mean(), 4)
)

print(

    "Log-verossimilhança média:",

    round(gmm_log_likelihood_val, 4)
)

In [ ]:
df_gmm_validation_uncertainty = pd.DataFrame({

    "Conjunto": [

        "Ajuste",

        "Validação"
    ],

    "Probabilidade máxima média": [

        gmm_confidence.mean(),

        gmm_confidence_val.mean()
    ],

    "Entropia média": [

        gmm_entropy.mean(),

        gmm_entropy_val.mean()
    ],

    "Log-verossimilhança média": [

        final_gmm.score(X_tuning_final),

        gmm_log_likelihood_val
    ],

    "Probabilidade inferior a 70% (%)": [

        100 * np.mean(gmm_confidence < 0.7),

        100 * np.mean(gmm_confidence_val < 0.7)
    ]
})

display(
    df_gmm_validation_uncertainty.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

# Intervalos iguais para comparação
bins_uncertainty = np.linspace(
    0,
    1,
    41
)

# Probabilidade máxima
axes[0].hist(

    gmm_confidence,

    bins=bins_uncertainty,

    density=True,

    alpha=0.5,

    color="steelblue",

    label="Ajuste"
)

axes[0].hist(

    gmm_confidence_val,

    bins=bins_uncertainty,

    density=True,

    alpha=0.5,

    color="darkorange",

    label="Validação"
)

axes[0].set_title(
    "Probabilidade Máxima"
)

axes[0].set_xlabel(
    "Probabilidade de pertencimento"
)

# Entropia
axes[1].hist(

    gmm_entropy,

    bins=bins_uncertainty,

    density=True,

    alpha=0.5,

    color="steelblue",

    label="Ajuste"
)

axes[1].hist(

    gmm_entropy_val,

    bins=bins_uncertainty,

    density=True,

    alpha=0.5,

    color="darkorange",

    label="Validação"
)

axes[1].set_title(
    "Entropia Normalizada"
)

axes[1].set_xlabel("Entropia")

for ax in axes:

    ax.set_xlim(0, 1)

    ax.set_ylabel("Densidade")

    ax.legend()

    ax.grid(alpha=0.3)

plt.suptitle(
    "Análise da Incerteza — Ajuste e Validação"
)

plt.tight_layout()
plt.show()

In [ ]:
# Distâncias aos centroides
kmeans_distances_tuning = final_kmeans.transform(

    X_tuning_final
)

kmeans_distances_val = final_kmeans.transform(

    X_val_final
)

# Distância ao centroide mais próximo
kmeans_min_distance_tuning = np.min(

    kmeans_distances_tuning,

    axis=1
)

kmeans_min_distance_val = np.min(

    kmeans_distances_val,

    axis=1
)

df_kmeans_distance_validation = pd.DataFrame({

    "Conjunto": [

        "Ajuste",

        "Validação"
    ],

    "Distância média": [

        kmeans_min_distance_tuning.mean(),

        kmeans_min_distance_val.mean()
    ],

    "Distância mediana": [

        np.median(kmeans_min_distance_tuning),

        np.median(kmeans_min_distance_val)
    ],

    "Percentil 95": [

        np.percentile(
            kmeans_min_distance_tuning,
            95
        ),

        np.percentile(
            kmeans_min_distance_val,
            95
        )
    ]
})

display(
    df_kmeans_distance_validation.round(4)
)

#### 10.8. Comparação visual da segmentação sísmica

In [ ]:
# Verificar os volumes previamente calculados
assert segmentation_volume.shape == seismic_data.shape

assert segmentation_volume_gmm.shape == seismic_data.shape

# Verificar os índices de validação
assert len(validation_indices) > 0

# Quantidade de traços para visualização
n_validation_display = min(

    MAX_DISPLAY_TRACES,

    len(validation_indices)
)

# Selecionar bloco central da região de validação
validation_offset = (

    len(validation_indices)

    - n_validation_display

) // 2

# Índices efetivamente utilizados
validation_display_indices = validation_indices[

    validation_offset:

    validation_offset + n_validation_display
]

# Amplitude original
amplitude_validation_detail = seismic_data[

    validation_display_indices,

    0,

    :
]

# Segmentação K-means
segmentation_validation_kmeans = segmentation_volume[

    validation_display_indices,

    0,

    :
]

# Segmentação GMM
segmentation_validation_gmm = segmentation_volume_gmm[

    validation_display_indices,

    0,

    :
]

print(

    "Primeiro traço:",

    validation_display_indices[0]
)

print(

    "Último traço:",

    validation_display_indices[-1]
)

print(

    "Quantidade de traços:",

    len(validation_display_indices)
)

print(

    "Dimensões do bloco:",

    amplitude_validation_detail.shape
)

In [ ]:
from matplotlib.colors import BoundaryNorm

# Limite de contraste da amplitude original
validation_v_limit = max(

    np.percentile(

        np.abs(amplitude_validation_detail),

        99.5
    ),

    1e-12
)

# Máscaras das regiões não classificadas
validation_kmeans_masked = np.ma.masked_equal(

    segmentation_validation_kmeans,

    -1
)

validation_gmm_masked = np.ma.masked_equal(

    segmentation_validation_gmm,

    -1
)

# Mapas de cores
validation_kmeans_cmap = plt.get_cmap(

    "Spectral",

    K_IDEAL

).copy()

validation_gmm_cmap = plt.get_cmap(

    "Spectral",

    GMM_IDEAL

).copy()

validation_kmeans_cmap.set_bad(
    "lightgray"
)

validation_gmm_cmap.set_bad(
    "lightgray"
)

# Normalização das cores categóricas
validation_kmeans_norm = BoundaryNorm(

    np.arange(-0.5, K_IDEAL + 0.5, 1),

    K_IDEAL
)

validation_gmm_norm = BoundaryNorm(

    np.arange(-0.5, GMM_IDEAL + 0.5, 1),

    GMM_IDEAL
)

# Criação da figura
fig, axes = plt.subplots(

    1, 3,

    figsize=(21, 7),

    sharey=True
)

# 1. Amplitude original
im_amp = axes[0].imshow(

    amplitude_validation_detail.T,

    cmap="seismic",

    vmin=-validation_v_limit,

    vmax=validation_v_limit,

    aspect="auto",

    interpolation="nearest"
)

axes[0].set_title(
    "Amplitude Sísmica Original"
)

# 2. Segmentação K-means
im_kmeans = axes[1].imshow(

    validation_kmeans_masked.T,

    cmap=validation_kmeans_cmap,

    norm=validation_kmeans_norm,

    aspect="auto",

    interpolation="nearest"
)

axes[1].set_title(

    f"K-means (K={K_IDEAL})"
)

# 3. Segmentação GMM
im_gmm = axes[2].imshow(

    validation_gmm_masked.T,

    cmap=validation_gmm_cmap,

    norm=validation_gmm_norm,

    aspect="auto",

    interpolation="nearest"
)

axes[2].set_title(

    f"GMM (K={GMM_IDEAL})"
)

# Eixos
for ax in axes:

    ax.set_xlabel(
        "Índice do traço no bloco"
    )

axes[0].set_ylabel(
    "Amostra temporal"
)

# Barras de cores
fig.colorbar(

    im_amp,

    ax=axes[0],

    fraction=0.046,

    label="Amplitude"
)

fig.colorbar(

    im_kmeans,

    ax=axes[1],

    fraction=0.046,

    ticks=np.arange(K_IDEAL),

    label="Cluster"
)

fig.colorbar(

    im_gmm,

    ax=axes[2],

    fraction=0.046,

    ticks=np.arange(GMM_IDEAL),

    label="Componente"
)

plt.suptitle(

    "Comparação da Segmentação Sísmica — Validação Interna",

    fontsize=16
)

plt.tight_layout()
plt.show()

### 11. Generalização entre Bacias Sedimentares

Nesta etapa, é investigada a capacidade de generalização
dos algoritmos K-means e Gaussian Mixture Model (GMM)
para dados sísmicos de uma bacia sedimentar diferente
daquela utilizada no treinamento.


#### 11.2. Preparação dos dados da Bacia de Campos

Os dados sísmicos da Bacia de Campos são preparados
utilizando os mesmos procedimentos empregados na
Bacia do Jequitinhonha.

In [ ]:
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

import time

# Verificar os atributos
assert selected_features == [
    "Amplitude",
    "Envelope",
    "Fase"
]

# Verificar os padronizadores
assert scaler.n_features_in_ == 7

assert scaler_original.n_features_in_ == 3

assert final_preprocessor is scaler_original

# Verificar os modelos treinados
assert final_kmeans.n_features_in_ == 3

assert final_gmm.n_features_in_ == 3

assert final_gmm.converged_

# Verificar os dados de Campos
assert seismic_data_campos.ndim == 3

assert seismic_data_campos.shape[1] == 1

assert dt_campos > 0

print("Dados e modelos verificados!")

print("\nDimensões de Campos:")
print(seismic_data_campos.shape)

print("\nIntervalo de amostragem temporal:")
print(dt_campos, "ms")

print("\nAtributos utilizados:")
print(selected_features)

print("\nClusters do K-means:", K_IDEAL)

print("Componentes do GMM:", GMM_IDEAL)

In [ ]:
# Utilizar todos os traços de Campos como universo amostral
campos_indices = np.arange(
    seismic_data_campos.shape[0]
)

# Construção da matriz de atributos
X_test = build_sample(
    seismic_data_campos,
    campos_indices,
    dt_campos,
    MAX_VALIDATION_SAMPLES,
    seed=RANDOM_STATE + 3
)

print("Amostras de Campos preparadas!")

print("\nDimensões da matriz:")
print(X_test.shape)

print("\nAtributos disponíveis:")
print(all_feature_names)

In [ ]:
# Selecionar amplitude, envelope e fase
X_test_original = X_test[:, selected_indices]

# Aplicar o padronizador do Jequitinhonha
X_test_final = final_preprocessor.transform(
    X_test_original
)

# Verificações
assert X_test_final.shape[1] == 3

assert len(X_test_final) == len(X_test)

assert np.isfinite(X_test_final).all()

print("Pré-processamento de Campos concluído!")

print("\nAntes da padronização:")
print(X_test_original.shape)

print("\nDepois da padronização:")
print(X_test_final.shape)

print("\nAtributos:")
print(selected_features)

In [ ]:
df_basin_attributes = pd.DataFrame({

    "Atributo": selected_features,

    "Média Jequitinhonha":
        X_val_original.mean(axis=0),

    "Média Campos":
        X_test_original.mean(axis=0),

    "Desvio Jequitinhonha":
        X_val_original.std(axis=0),

    "Desvio Campos":
        X_test_original.std(axis=0)
})

display(
    df_basin_attributes.round(4)
)

#### 11.3. Aplicação dos modelos treinados

Os modelos K-means e GMM ajustados com os dados
da Bacia do Jequitinhonha são aplicados às amostras
sísmicas da Bacia de Campos.

Nenhum parâmetro dos modelos é reajustado durante
essa aplicação.

In [ ]:
# Predição do K-means
labels_test_kmeans = final_kmeans.predict(
    X_test_final
)

# Predição do GMM
labels_test_gmm = final_gmm.predict(
    X_test_final
)

# Verificações
assert len(labels_test_kmeans) == len(X_test_final)

assert len(labels_test_gmm) == len(X_test_final)

print("Predições realizadas com sucesso!")

print("\nK-means:")

print(
    "Clusters encontrados:",
    np.unique(labels_test_kmeans)
)

print("\nGMM:")

print(
    "Componentes encontradas:",
    np.unique(labels_test_gmm)
)

print("\nTotal de amostras:")
print(len(X_test_final))

In [ ]:
# Gerador aleatório específico do teste externo
test_rng = np.random.default_rng(
    RANDOM_STATE + 20
)

# Quantidade equivalente à validação interna
n_test_evaluation = min(
    SILHOUETTE_SAMPLES,
    len(X_validation_evaluation),
    len(X_test_final)
)

# Selecionar amostras sem reposição
test_evaluation_indices = test_rng.choice(

    len(X_test_final),

    size=n_test_evaluation,

    replace=False
)

# Matriz de avaliação
X_test_evaluation = X_test_final[
    test_evaluation_indices
]

# Rótulos correspondentes
labels_test_kmeans_eval = labels_test_kmeans[
    test_evaluation_indices
]

labels_test_gmm_eval = labels_test_gmm[
    test_evaluation_indices
]

print("Total de amostras de Campos:")
print(len(X_test_final))

print("\nAmostras utilizadas nas métricas:")
print(len(X_test_evaluation))

#### 11.4. Comparação quantitativa entre as bacias

In [ ]:
# Avaliação do K-means
metrics_test_kmeans = evaluate_clustering(

    X_test_evaluation,

    labels_test_kmeans_eval
)

# Avaliação do GMM
metrics_test_gmm = evaluate_clustering(

    X_test_evaluation,

    labels_test_gmm_eval
)

# Tabela do teste externo
df_external_test = pd.DataFrame([

    {
        "Modelo": "K-means",
        "K": K_IDEAL,
        **metrics_test_kmeans
    },

    {
        "Modelo": "GMM",
        "K": GMM_IDEAL,
        **metrics_test_gmm
    }
])

display(
    df_external_test.round(4)
)

In [ ]:
df_generalization_results = pd.concat([

    df_internal_validation.assign(
        Bacia="Jequitinhonha"
    ),

    df_external_test.assign(
        Bacia="Campos"
    )

], ignore_index=True)

# Organizar as colunas
df_generalization_results = df_generalization_results[[

    "Bacia",

    "Modelo",

    "K",

    "Silhouette",

    "Davies-Bouldin",

    "Calinski-Harabasz",

    "Clusters encontrados"
]]

display(
    df_generalization_results.round(4)
)

In [ ]:
generalization_changes = []

for model_name in ["K-means", "GMM"]:

    # Resultados no Jequitinhonha
    internal = df_internal_validation.loc[
        df_internal_validation["Modelo"] == model_name
    ].iloc[0]

    # Resultados em Campos
    external = df_external_test.loc[
        df_external_test["Modelo"] == model_name
    ].iloc[0]

    generalization_changes.append({

        "Modelo": model_name,

        "Silhouette Jequitinhonha":
            internal["Silhouette"],

        "Silhouette Campos":
            external["Silhouette"],

        "Variação Silhouette":
            external["Silhouette"]
            - internal["Silhouette"],

        "Variação Davies-Bouldin":
            external["Davies-Bouldin"]
            - internal["Davies-Bouldin"],

        "Variação Calinski-Harabasz":
            external["Calinski-Harabasz"]
            - internal["Calinski-Harabasz"]
    })

df_generalization_changes = pd.DataFrame(
    generalization_changes
)

display(
    df_generalization_changes.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(17, 5)
)

metrics_generalization = [

    ("Silhouette", "Silhouette"),

    ("Davies-Bouldin", "Davies-Bouldin"),

    ("Calinski-Harabasz", "Calinski-Harabasz")
]

for ax, (metric, title) in zip(
    axes,
    metrics_generalization
):

    df_plot = df_generalization_results.pivot(

        index="Modelo",

        columns="Bacia",

        values=metric

    ).reindex(["K-means", "GMM"])

    df_plot = df_plot[[
        "Jequitinhonha",
        "Campos"
    ]]

    df_plot.plot(

        kind="bar",

        ax=ax,

        color=[
            "steelblue",
            "darkorange"
        ],

        width=0.7
    )

    ax.set_title(title)

    ax.set_xlabel("Modelo")

    ax.set_ylabel(metric)

    ax.tick_params(
        axis="x",
        rotation=0
    )

    ax.margins(y=0.15)

    for container in ax.containers:

        ax.bar_label(
            container,
            fmt="%.3g",
            padding=3
        )

    ax.grid(
        axis="y",
        alpha=0.3
    )

plt.suptitle(
    "Comparação das Métricas entre as Bacias",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Aplicar os modelos com o mesmo K
labels_test_kmeans_common = kmeans_common.predict(
    X_test_final
)

labels_test_gmm_common = gmm_common.predict(
    X_test_final
)

# Avaliar K-means
metrics_test_kmeans_common = evaluate_clustering(

    X_test_evaluation,

    labels_test_kmeans_common[
        test_evaluation_indices
    ]
)

# Avaliar GMM
metrics_test_gmm_common = evaluate_clustering(

    X_test_evaluation,

    labels_test_gmm_common[
        test_evaluation_indices
    ]
)

# Tabela de Campos
df_common_k_external = pd.DataFrame([

    {
        "Modelo": "K-means",
        "K": K_COMMON,
        **metrics_test_kmeans_common
    },

    {
        "Modelo": "GMM",
        "K": K_COMMON,
        **metrics_test_gmm_common
    }
])

# Comparação entre bacias
df_common_k_generalization = pd.concat([

    df_common_k_validation.assign(
        Bacia="Jequitinhonha"
    ),

    df_common_k_external.assign(
        Bacia="Campos"
    )

], ignore_index=True)

display(
    df_common_k_generalization.round(4)
)

#### 11.5. Distribuição dos agrupamentos entre as bacias

A distribuição percentual dos agrupamentos é comparada
entre a validação interna da Bacia do Jequitinhonha
e o teste externo da Bacia de Campos.

In [ ]:
df_generalization_distribution_kmeans = pd.DataFrame({

    "Cluster": np.arange(K_IDEAL),

    "Jequitinhonha (%)":
        calculate_cluster_distribution(
            labels_val_kmeans,
            K_IDEAL
        ),

    "Campos (%)":
        calculate_cluster_distribution(
            labels_test_kmeans,
            K_IDEAL
        )
})

display(
    df_generalization_distribution_kmeans.round(4)
)

In [ ]:
df_generalization_distribution_gmm = pd.DataFrame({

    "Componente": np.arange(GMM_IDEAL),

    "Jequitinhonha (%)":
        calculate_cluster_distribution(
            labels_val_gmm,
            GMM_IDEAL
        ),

    "Campos (%)":
        calculate_cluster_distribution(
            labels_test_gmm,
            GMM_IDEAL
        )
})

display(
    df_generalization_distribution_gmm.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 6),
    sharey=True
)

# K-means
df_generalization_distribution_kmeans.set_index(

    "Cluster"

).plot(

    kind="bar",

    ax=axes[0],

    color=[
        "steelblue",
        "darkorange"
    ]
)

axes[0].set_title(
    f"Distribuição K-means (K={K_IDEAL})"
)

axes[0].set_xlabel("Cluster")

# GMM
df_generalization_distribution_gmm.set_index(

    "Componente"

).plot(

    kind="bar",

    ax=axes[1],

    color=[
        "steelblue",
        "darkorange"
    ]
)

axes[1].set_title(
    f"Distribuição GMM (K={GMM_IDEAL})"
)

axes[1].set_xlabel("Componente")

# Configuração dos eixos
for ax in axes:

    ax.set_ylabel(
        "Percentual de amostras (%)"
    )

    ax.tick_params(
        axis="x",
        rotation=0
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.legend()

plt.suptitle(
    "Distribuição dos Agrupamentos entre as Bacias",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
# Mudança de distribuição do K-means
tv_kmeans_external = total_variation_distance(

    df_generalization_distribution_kmeans[
        "Jequitinhonha (%)"
    ],

    df_generalization_distribution_kmeans[
        "Campos (%)"
    ]
)

# Mudança de distribuição do GMM
tv_gmm_external = total_variation_distance(

    df_generalization_distribution_gmm[
        "Jequitinhonha (%)"
    ],

    df_generalization_distribution_gmm[
        "Campos (%)"
    ]
)

df_generalization_shift = pd.DataFrame({

    "Modelo": [
        "K-means",
        "GMM"
    ],

    "Variação entre bacias": [

        tv_kmeans_external,

        tv_gmm_external
    ]
})

display(
    df_generalization_shift.round(4)
)

#### 11.6. Análise da generalização e das incertezas

A capacidade de generalização também é investigada
por meio de indicadores específicos de cada algoritmo.

No K-means, são analisadas as distâncias das amostras
aos centroides mais próximos.

No GMM, são avaliadas as probabilidades posteriores,
a entropia normalizada e a log-verossimilhança média.

In [ ]:
# Distâncias aos centroides aprendidos
kmeans_distances_test = final_kmeans.transform(
    X_test_final
)

# Distância ao centroide mais próximo
kmeans_min_distance_test = np.min(

    kmeans_distances_test,

    axis=1
)

# Comparação entre bacias
df_kmeans_generalization_distance = pd.DataFrame({

    "Bacia": [

        "Jequitinhonha",

        "Campos"
    ],

    "Distância média": [

        kmeans_min_distance_val.mean(),

        kmeans_min_distance_test.mean()
    ],

    "Distância mediana": [

        np.median(kmeans_min_distance_val),

        np.median(kmeans_min_distance_test)
    ],

    "Percentil 95": [

        np.percentile(
            kmeans_min_distance_val,
            95
        ),

        np.percentile(
            kmeans_min_distance_test,
            95
        )
    ]
})

display(
    df_kmeans_generalization_distance.round(4)
)

In [ ]:
# Probabilidades de pertencimento
gmm_probabilities_test = final_gmm.predict_proba(
    X_test_final
)

# Probabilidade máxima
gmm_confidence_test = np.max(

    gmm_probabilities_test,

    axis=1
)

# Entropia normalizada
gmm_entropy_test = (

    -np.sum(

        gmm_probabilities_test

        * np.log(

            np.clip(
                gmm_probabilities_test,
                1e-12,
                1
            )
        ),

        axis=1

    ) / np.log(GMM_IDEAL)
)

# Corrigir pequenos desvios numéricos
gmm_entropy_test = np.clip(
    gmm_entropy_test,
    0,
    1
)

# Log-verossimilhança média
gmm_log_likelihood_test = final_gmm.score(
    X_test_final
)

# Log-verossimilhança individual
gmm_log_density_test = final_gmm.score_samples(
    X_test_final
)

gmm_log_density_val = final_gmm.score_samples(
    X_val_final
)

# Verificar probabilidades
assert np.allclose(
    gmm_probabilities_test.sum(axis=1),
    1
)

print("Indicadores probabilísticos calculados!")

In [ ]:
df_gmm_generalization_uncertainty = pd.DataFrame({

    "Bacia": [

        "Jequitinhonha",

        "Campos"
    ],

    "Probabilidade máxima média": [

        gmm_confidence_val.mean(),

        gmm_confidence_test.mean()
    ],

    "Entropia média": [

        gmm_entropy_val.mean(),

        gmm_entropy_test.mean()
    ],

    "Log-verossimilhança média": [

        gmm_log_likelihood_val,

        gmm_log_likelihood_test
    ],

    "Probabilidade inferior a 70% (%)": [

        100 * np.mean(
            gmm_confidence_val < 0.7
        ),

        100 * np.mean(
            gmm_confidence_test < 0.7
        )
    ]
})

display(
    df_gmm_generalization_uncertainty.round(4)
)

In [ ]:
# Limite de distância do K-means
kmeans_distance_threshold = np.percentile(
    kmeans_min_distance_val,
    95
)

# Limite inferior da densidade do GMM
gmm_density_threshold = np.percentile(
    gmm_log_density_val,
    5
)

# Amostras externas acima do limite de distância
kmeans_atypical_test = (

    kmeans_min_distance_test
    > kmeans_distance_threshold
)

# Amostras externas abaixo do limite de densidade
gmm_atypical_test = (

    gmm_log_density_test
    < gmm_density_threshold
)

df_external_atypical = pd.DataFrame({

    "Modelo": [
        "K-means",
        "GMM"
    ],

    "Amostras atípicas no Jequitinhonha (%)": [

        100 * np.mean(
            kmeans_min_distance_val
            > kmeans_distance_threshold
        ),

        100 * np.mean(
            gmm_log_density_val
            < gmm_density_threshold
        )
    ],

    "Amostras atípicas em Campos (%)": [

        100 * kmeans_atypical_test.mean(),

        100 * gmm_atypical_test.mean()
    ]
})

display(
    df_external_atypical.round(4)
)

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

# Distância aos centroides
axes[0].hist(

    kmeans_min_distance_val,

    bins=50,

    density=True,

    alpha=0.55,

    color="steelblue",

    label="Jequitinhonha"
)

axes[0].hist(

    kmeans_min_distance_test,

    bins=50,

    density=True,

    alpha=0.55,

    color="darkorange",

    label="Campos"
)

axes[0].axvline(
    kmeans_distance_threshold,
    color="red",
    linestyle="--",
    label="Limite de referência"
)

axes[0].set_title(
    "K-means — Distância aos Centroides"
)

axes[0].set_xlabel("Distância")

# Log-densidade do GMM
axes[1].hist(

    gmm_log_density_val,

    bins=50,

    density=True,

    alpha=0.55,

    color="steelblue",

    label="Jequitinhonha"
)

axes[1].hist(

    gmm_log_density_test,

    bins=50,

    density=True,

    alpha=0.55,

    color="darkorange",

    label="Campos"
)

axes[1].axvline(
    gmm_density_threshold,
    color="red",
    linestyle="--",
    label="Limite de referência"
)

axes[1].set_title(
    "GMM — Log-densidade"
)

axes[1].set_xlabel("Log-densidade")

for ax in axes:

    ax.set_ylabel("Densidade")

    ax.legend()

    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Concordância em Campos
ari_test = adjusted_rand_score(

    labels_test_kmeans,

    labels_test_gmm
)

nmi_test = normalized_mutual_info_score(

    labels_test_kmeans,

    labels_test_gmm
)

# Comparação entre bacias
df_generalization_agreement = pd.DataFrame({

    "Bacia": [

        "Jequitinhonha",

        "Campos"
    ],

    "ARI": [

        ari_validation,

        ari_test
    ],

    "NMI": [

        nmi_validation,

        nmi_test
    ]
})

display(
    df_generalization_agreement.round(4)
)

#### 11.7. Segmentação sísmica da Bacia de Campos

Os modelos treinados exclusivamente com os dados
da Bacia do Jequitinhonha são aplicados aos registros
sísmicos completos da Bacia de Campos.

A segmentação é realizada em blocos de traços,
utilizando os mesmos procedimentos de normalização,
extração e padronização dos atributos.

In [ ]:
print("--- Segmentação de Campos pelo K-means ---")

start_time = time.perf_counter()

segmentation_volume_campos_kmeans = segment_seismic_data(

    seismic_data_campos,

    dt_campos,

    final_kmeans,

    final_preprocessor,

    selected_indices
)

kmeans_campos_segmentation_time = (

    time.perf_counter() - start_time
)

# Verificação
assert (

    segmentation_volume_campos_kmeans.shape

    == seismic_data_campos.shape
)

print("\nSegmentação concluída!")

print(
    "Dimensões:",
    segmentation_volume_campos_kmeans.shape
)

print(

    f"Tempo de segmentação: "

    f"{kmeans_campos_segmentation_time:.2f} segundos"
)

In [ ]:
print("--- Segmentação de Campos pelo GMM ---")

start_time = time.perf_counter()

segmentation_volume_campos_gmm = segment_seismic_data(

    seismic_data_campos,

    dt_campos,

    final_gmm,

    final_preprocessor,

    selected_indices
)

gmm_campos_segmentation_time = (

    time.perf_counter() - start_time
)

# Verificação
assert (

    segmentation_volume_campos_gmm.shape

    == seismic_data_campos.shape
)

print("\nSegmentação concluída!")

print(
    "Dimensões:",
    segmentation_volume_campos_gmm.shape
)

print(

    f"Tempo de segmentação: "

    f"{gmm_campos_segmentation_time:.2f} segundos"
)

In [ ]:
# Rótulos em uma dimensão
cluster_labels_flat_campos_kmeans = (
    segmentation_volume_campos_kmeans.reshape(-1)
)

cluster_labels_flat_campos_gmm = (
    segmentation_volume_campos_gmm.reshape(-1)
)

# Selecionar rótulos válidos
valid_campos_kmeans = cluster_labels_flat_campos_kmeans[
    cluster_labels_flat_campos_kmeans >= 0
]

valid_campos_gmm = cluster_labels_flat_campos_gmm[
    cluster_labels_flat_campos_gmm >= 0
]

# Verificar amostras válidas
assert len(valid_campos_kmeans) > 0

assert len(valid_campos_gmm) > 0

assert len(valid_campos_kmeans) == len(valid_campos_gmm)

# Distribuição do K-means
df_campos_kmeans_distribution = pd.DataFrame({

    "Cluster": np.arange(K_IDEAL),

    "Quantidade": np.bincount(
        valid_campos_kmeans,
        minlength=K_IDEAL
    ),

    "% Volume Classificado":
        calculate_cluster_distribution(
            valid_campos_kmeans,
            K_IDEAL
        )
})

# Distribuição do GMM
df_campos_gmm_distribution = pd.DataFrame({

    "Componente": np.arange(GMM_IDEAL),

    "Quantidade": np.bincount(
        valid_campos_gmm,
        minlength=GMM_IDEAL
    ),

    "% Volume Classificado":
        calculate_cluster_distribution(
            valid_campos_gmm,
            GMM_IDEAL
        )
})

print("Distribuição completa — K-means:")

display(
    df_campos_kmeans_distribution.round(4)
)

print("\nDistribuição completa — GMM:")

display(
    df_campos_gmm_distribution.round(4)
)

print("\nPercentual de amostras classificadas:")

print(

    round(

        100 * len(valid_campos_kmeans)

        / seismic_data_campos.size,

        4
    ),

    "%"
)

#### 11.8. Comparação visual da segmentação sísmica

In [ ]:
from matplotlib.colors import BoundaryNorm

# Dados originais em duas dimensões
seismic_data_campos_2d = seismic_data_campos[:, 0, :]

# Segmentação K-means em duas dimensões
segmentation_campos_kmeans_2d = (
    segmentation_volume_campos_kmeans[:, 0, :]
)

# Segmentação GMM em duas dimensões
segmentation_campos_gmm_2d = (
    segmentation_volume_campos_gmm[:, 0, :]
)

# Mapas de cores
campos_kmeans_cmap = plt.get_cmap(
    "Spectral",
    K_IDEAL
).copy()

campos_gmm_cmap = plt.get_cmap(
    "Spectral",
    GMM_IDEAL
).copy()

# Regiões não classificadas
campos_kmeans_cmap.set_bad("lightgray")

campos_gmm_cmap.set_bad("lightgray")

# Normalização categórica
campos_kmeans_norm = BoundaryNorm(

    np.arange(-0.5, K_IDEAL + 0.5, 1),

    K_IDEAL
)

campos_gmm_norm = BoundaryNorm(

    np.arange(-0.5, GMM_IDEAL + 0.5, 1),

    GMM_IDEAL
)

print("Mapas preparados para visualização!")

In [ ]:
def plot_campos_comparison(
    start=0,
    stop=None,
    title="Segmentação Sísmica — Bacia de Campos"
):

    if stop is None:
        stop = seismic_data_campos_2d.shape[0]

    # Selecionar a região
    amplitude = seismic_data_campos_2d[
        start:stop
    ]

    segmentation_kmeans = np.ma.masked_equal(

        segmentation_campos_kmeans_2d[
            start:stop
        ],

        -1
    )

    segmentation_gmm = np.ma.masked_equal(

        segmentation_campos_gmm_2d[
            start:stop
        ],

        -1
    )

    # Contraste da amplitude original
    v_limit_campos = max(

        np.percentile(
            np.abs(amplitude),
            99.5
        ),

        1e-12
    )

    # Eixo temporal em milissegundos
    extent = (

        start - 0.5,

        stop - 0.5,

        (amplitude.shape[1] - 0.5) * dt_campos,

        -0.5 * dt_campos
    )

    # Organização dos gráficos
    fig, axes = plt.subplots(

        1, 3,

        figsize=(21, 7),

        sharey=True
    )

    # 1. Amplitude original
    im_amp = axes[0].imshow(

        amplitude.T,

        cmap="seismic",

        vmin=-v_limit_campos,

        vmax=v_limit_campos,

        extent=extent,

        aspect="auto",

        interpolation="nearest"
    )

    axes[0].set_title(
        "Amplitude Sísmica Original"
    )

    # 2. Segmentação K-means
    im_kmeans = axes[1].imshow(

        segmentation_kmeans.T,

        cmap=campos_kmeans_cmap,

        norm=campos_kmeans_norm,

        extent=extent,

        aspect="auto",

        interpolation="nearest"
    )

    axes[1].set_title(
        f"K-means (K={K_IDEAL})"
    )

    # 3. Segmentação GMM
    im_gmm = axes[2].imshow(

        segmentation_gmm.T,

        cmap=campos_gmm_cmap,

        norm=campos_gmm_norm,

        extent=extent,

        aspect="auto",

        interpolation="nearest"
    )

    axes[2].set_title(
        f"GMM (K={GMM_IDEAL})"
    )

    # Configuração dos eixos
    for ax in axes:

        ax.set_xlabel("Índice do traço")

    axes[0].set_ylabel(
        "Tempo relativo à primeira amostra (ms)"
    )

    # Barras de cores
    fig.colorbar(

        im_amp,

        ax=axes[0],

        fraction=0.046,

        label="Amplitude"
    )

    fig.colorbar(

        im_kmeans,

        ax=axes[1],

        ticks=np.arange(K_IDEAL),

        fraction=0.046,

        label="Cluster"
    )

    fig.colorbar(

        im_gmm,

        ax=axes[2],

        ticks=np.arange(GMM_IDEAL),

        fraction=0.046,

        label="Componente"
    )

    plt.suptitle(
        title,
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

plot_campos_comparison()

In [ ]:
# Quantidade de traços para visualização
campos_n_display = min(

    MAX_DISPLAY_TRACES,

    seismic_data_campos.shape[0]
)

# Limites do bloco central
campos_display_start = (

    seismic_data_campos.shape[0]

    - campos_n_display

) // 2

campos_display_stop = (

    campos_display_start

    + campos_n_display
)

# Visualização detalhada
plot_campos_comparison(

    start=campos_display_start,

    stop=campos_display_stop,

    title="Segmentação Sísmica de Campos — Detalhe"
)

### 12. Análise de Desempenho Computacional

O desempenho computacional dos algoritmos K-means
e GMM é avaliado considerando o tempo de treinamento
e o tempo de segmentação dos dados sísmicos das
bacias do Jequitinhonha e de Campos.

In [ ]:
df_computational_performance = pd.DataFrame({

    "Modelo": ["K-means", "GMM"],

    "Número de agrupamentos": [
        K_IDEAL,
        GMM_IDEAL
    ],

    "Treinamento (s)": [
        kmeans_training_time,
        gmm_training_time
    ],

    "Segmentação Jequitinhonha (s)": [
        kmeans_segmentation_time,
        gmm_segmentation_time
    ],

    "Segmentação Campos (s)": [
        kmeans_campos_segmentation_time,
        gmm_campos_segmentation_time
    ]
})

display(
    df_computational_performance.round(2)
)

#### 12.2. Comparação gráfica do desempenho


In [ ]:
# Selecionar os tempos de execução
df_performance_plot = (
    df_computational_performance
    .set_index("Modelo")
    .drop(columns="Número de agrupamentos")
    .T
)

# Gráfico comparativo
ax = df_performance_plot.plot(
    kind="bar",
    figsize=(12, 6),
    color=["steelblue", "darkorange"],
    width=0.7
)

# Exibir os valores
for container in ax.containers:

    ax.bar_label(
        container,
        fmt="%.2f",
        padding=3
    )

plt.title(
    "Desempenho Computacional — K-means e GMM"
)

plt.ylabel("Tempo de execução (s)")

plt.xlabel("")

plt.xticks(rotation=0)

plt.legend(title="Modelo")

plt.grid(axis="y", alpha=0.3)

ax.margins(y=0.15)

plt.tight_layout()
plt.show()